# IMTM historico e comparacao MAD/ZCA-Ledoit-Wolf

**Papel nesta entrega:** Suplementar: analise historica descrita no trabalho

**Origem:** https://colab.research.google.com/drive/12cUs6rcawPBvQ23WM6gzsaJjjqksHLez

Copia de publicacao organizada em 7 out. 2026. Saidas, contadores e metadados de sessao foram removidos; isso nao significa que a campanha original nao tenha sido executada. Nenhum resultado novo foi produzido na curadoria. O registro de proveniencia relaciona codigo, fonte e alteracoes de configuracao.

Dependencias e caminhos sao os da campanha historica, nao das quatro execucoes padronizadas. Esses cadernos nao devem substituir os resultados principais. Consulte o mapa de cobertura e mantenha separadas as datas de execucao. A comparacao de sphering esta nas ultimas quatro celulas de codigo (indices originais 16-19). Os caminhos de retomada entre as fases estao congelados; nao e um fluxo portavel que automaticamente encadeia uma nova campanha de ponta a ponta. Leia os seletores antes de executar; o objetivo primario aqui e preservar o codigo efetivamente usado.

In [ ]:
%pip install -q "polars>=1.0,<2" requests

from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import re

import polars as pl
from IPython.display import display

# ============================================================
# 1. Localização dos dados
# ============================================================

RAIZ = Path("/content/drive/MyDrive/mba_tcc")
ENTRADA = RAIZ / "IMTM_HepG2"

if not ENTRADA.is_dir():
    raise FileNotFoundError(f"Pasta não encontrada:\n{ENTRADA}")

arquivos = sorted(
    ENTRADA.glob("*_CP_Profiles_Aggregated.csv")
)

if not arquivos:
    raise FileNotFoundError(
        f"Nenhum CSV agregado foi encontrado em:\n{ENTRADA}"
    )

# ============================================================
# 2. Seleção explícita das versões corrigidas
# ============================================================

arquivo_antigo = (
    "2023-07-07_HepG2_10uM_B1001_R4_CP_Profiles_Aggregated.csv"
)

arquivos_corrigidos = {
    "2023-07-17_HepG2_10uM_B1001_R4_CP_Profiles_Aggregated.csv",
    "2023-07-07_HepG2_10uM_B1007_R4_CP_Profiles_Aggregated.csv",
}

nomes_encontrados = {p.name for p in arquivos}
faltam_corrigidos = arquivos_corrigidos - nomes_encontrados

if faltam_corrigidos:
    raise ValueError(
        "Os arquivos corrigidos de R4 precisam estar presentes:\n"
        + "\n".join(sorted(faltam_corrigidos))
    )

# Não apagar o arquivo antigo: apenas não incorporá-lo.
selecionados = [
    p for p in arquivos
    if p.name != arquivo_antigo
]

ignorados = [
    p.name for p in arquivos
    if p.name == arquivo_antigo
]

# ============================================================
# 3. Conferir as 7 placas × 4 réplicas
# ============================================================

padrao = re.compile(
    r"^\d{4}-\d{2}-\d{2}_HepG2_10uM_"
    r"(B100[1-7])_(R[1-4])_CP_Profiles_Aggregated\.csv$"
)

por_chave = {}

for caminho in selecionados:
    correspondencia = padrao.fullmatch(caminho.name)

    if correspondencia is None:
        raise ValueError(
            f"Nome de arquivo inesperado:\n{caminho.name}"
        )

    placa, replica = correspondencia.groups()
    chave = (placa, replica)

    if chave in por_chave:
        raise ValueError(
            f"Mais de um arquivo para {placa}, {replica}.\n"
            "A importação foi interrompida para evitar duplicação."
        )

    por_chave[chave] = caminho

chaves_esperadas = {
    (f"B100{placa}", f"R{replica}")
    for placa in range(1, 8)
    for replica in range(1, 5)
}

if set(por_chave) != chaves_esperadas:
    raise ValueError(
        "O conjunto de placas está incompleto.\n"
        f"Faltam: {sorted(chaves_esperadas - set(por_chave))}"
    )

# ============================================================
# 4. Ler e conferir cada arquivo
# ============================================================

META_ORIGINAIS = [
    "Metadata_Batch",
    "Metadata_Plate",
    "Metadata_Well",
    "Metadata_Object_Count",
]

partes = []
registro = []
colunas_referencia = None

for numero, ((placa, replica), caminho) in enumerate(
    sorted(por_chave.items()),
    start=1,
):
    with caminho.open(encoding="utf-8-sig", newline="") as arquivo:
        cabecalho = next(csv.reader(arquivo))

    if len(cabecalho) != len(set(cabecalho)):
        raise ValueError(
            f"Cabeçalho com nomes repetidos:\n{caminho.name}"
        )

    features = [
        coluna
        for coluna in cabecalho
        if coluna.startswith(("Nuc_", "Cells_", "Cyto_"))
    ]

    # Impedir que colunas inesperadas sejam ignoradas.
    if not features or set(cabecalho) != set(META_ORIGINAIS + features):
        raise ValueError(
            f"Estrutura diferente da esperada:\n{caminho.name}\n"
            "Precisamos revisar o cabeçalho antes de continuar."
        )

    if colunas_referencia is None:
        colunas_referencia = cabecalho
        FEATURES_IMTM = features

    elif set(cabecalho) != set(colunas_referencia):
        raise ValueError(
            f"O conjunto de colunas difere entre arquivos:\n"
            f"{caminho.name}"
        )

    tipos = {coluna: pl.Float64 for coluna in features}
    tipos.update({
        "Metadata_Batch": pl.String,
        "Metadata_Plate": pl.String,
        "Metadata_Well": pl.String,
        "Metadata_Object_Count": pl.Float64,
    })

    tabela = (
        pl.read_csv(
            caminho,
            schema_overrides=tipos,
            ignore_errors=False,
        )
        .select(colunas_referencia)
    )

    if tabela.is_empty():
        raise ValueError(f"Arquivo vazio: {caminho.name}")

    # Conferir a identidade interna, sem sobrescrevê-la.
    placas_internas = tabela["Metadata_Plate"].unique().to_list()

    if (
        tabela["Metadata_Plate"].null_count() > 0
        or placas_internas != [f"{placa}_{replica}"]
    ):
        raise ValueError(
            f"A identificação interna difere do nome do arquivo:\n"
            f"{caminho.name}\n"
            f"Metadata_Plate: {placas_internas}"
        )

    pocos = tabela["Metadata_Well"]

    if pocos.null_count() > 0 or pocos.n_unique() != tabela.height:
        raise ValueError(
            f"Poços nulos ou repetidos em:\n{caminho.name}"
        )

    if not pocos.str.contains(
        r"^[A-P](0[1-9]|1[0-9]|2[0-4])$"
    ).all():
        raise ValueError(
            f"Identificador de poço inesperado em:\n{caminho.name}"
        )

    # Registrar a identidade exata dos arquivos utilizados.
    resumo_hash = hashlib.sha256()

    with caminho.open("rb") as arquivo:
        for bloco in iter(
            lambda: arquivo.read(1024 * 1024),
            b"",
        ):
            resumo_hash.update(bloco)

    registro.append({
        "arquivo": caminho.name,
        "placa": placa,
        "replica": replica,
        "n_pocos": tabela.height,
        "n_medidas": len(features),
        "sha256": resumo_hash.hexdigest(),
    })

    # Preservar todos os metadados originais e acrescentar
    # identificadores explícitos para o nosso processamento.
    tabela = tabela.with_columns(
        pl.lit(placa).alias("Metadata_Plate_ID"),
        pl.lit(replica).alias("Metadata_Replicate"),
        pl.lit(caminho.name).alias("Metadata_Source_File"),
    )

    partes.append(tabela)

    print(
        f"[{numero:02d}/28] {placa} {replica}: "
        f"{tabela.height} poços",
        flush=True,
    )

# ============================================================
# 5. Concatenar sem agregar novamente os poços
# ============================================================

df_imtm_raw = pl.concat(
    partes,
    how="vertical",
    rechunk=False,
)

CHAVES_IMTM = [
    "Metadata_Plate_ID",
    "Metadata_Replicate",
    "Metadata_Well",
]

if df_imtm_raw.select(CHAVES_IMTM).is_duplicated().any():
    raise RuntimeError(
        "Existem combinações placa–réplica–poço repetidas."
    )

registro_imtm = pl.DataFrame(registro)

# ============================================================
# 6. Salvar em uma pasta nova
# ============================================================

identificador = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S_%fZ"
)

PASTA_EXECUCAO = (
    RAIZ
    / "pipeline_IMTM_HepG2"
    / identificador
)

PASTA_EXECUCAO.mkdir(parents=True, exist_ok=False)

ARQUIVO_RAW = (
    PASTA_EXECUCAO
    / "imtm_hepg2_agregado_nao_normalizado.parquet"
)

df_imtm_raw.write_parquet(
    ARQUIVO_RAW,
    compression="zstd",
)

registro_imtm.write_csv(
    PASTA_EXECUCAO / "registro_entrada.csv",
)

# Conferir a estrutura gravada, sem carregar outra cópia completa.
salvo = pl.scan_parquet(ARQUIVO_RAW)

linhas_salvas = salvo.select(pl.len()).collect().item()
schema_salvo = salvo.collect_schema()

if (
    linhas_salvas != df_imtm_raw.height
    or list(schema_salvo.items())
    != list(df_imtm_raw.schema.items())
):
    raise RuntimeError(
        "Falha na conferência estrutural do Parquet salvo."
    )

manifesto = {
    "fonte": str(ENTRADA),
    "arquivos_ignorados_na_selecao": ignorados,
    "motivo": (
        "Arquivo B1001_R4 anterior à correção; "
        "preservado na pasta de origem."
    ),
    "normalizacao_aplicada": False,
    "features": FEATURES_IMTM,
    "versao_polars": pl.__version__,
}

(PASTA_EXECUCAO / "manifesto.json").write_text(
    json.dumps(manifesto, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\nIMPORTAÇÃO IMTM CONCLUÍDA")
print("=" * 65)
print(f"Arquivos utilizados: {len(registro)}")
print(f"Arquivos ignorados na seleção: {ignorados}")
print(f"Linhas: {df_imtm_raw.height:,}")
print(f"Medidas: {len(FEATURES_IMTM):,}")
print(f"Colunas totais: {df_imtm_raw.width:,}")
print(f"Parquet: {ARQUIVO_RAW}")

with pl.Config(tbl_rows=30, tbl_cols=5):
    display(
        registro_imtm.select(
            "placa", "replica", "n_pocos", "n_medidas"
        )
    )

print("\nTabela disponível em: df_imtm_raw")
print("Nenhuma normalização ou seleção de medidas foi aplicada.")
print("Nenhum arquivo de origem foi alterado.")

In [ ]:
from pathlib import Path, PurePosixPath
import hashlib
import json
import zipfile

import requests
import polars as pl

NOME_MAPA = "2023-08-14_Annotation2_IMTM_HepG2.csv"

# Endereço indicado no tutorial dos autores.
URL_ARQUIVO_AUTORES = (
    "https://zenodo.org/records/13309566/"
    "files/Profile_Analysis.zip?download=1"
)

PASTA_MAPA = PASTA_EXECUCAO / "mapa_experimental"
PASTA_MAPA.mkdir(exist_ok=True)

CAMINHO_MAPA = PASTA_MAPA / NOME_MAPA
ZIP_LOCAL = PASTA_MAPA / "Profile_Analysis_fonte.zip"

if not CAMINHO_MAPA.is_file():
    print("Obtendo o mapa experimental dos autores...", flush=True)

    # O arquivo temporário evita considerar um download interrompido
    # como se fosse um ZIP completo.
    temporario = ZIP_LOCAL.with_suffix(".zip.part")

    try:
        with requests.get(
            URL_ARQUIVO_AUTORES,
            stream=True,
            timeout=(20, 120),
        ) as resposta:
            resposta.raise_for_status()

            with temporario.open("wb") as arquivo:
                for bloco in resposta.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if bloco:
                        arquivo.write(bloco)

        if not zipfile.is_zipfile(temporario):
            raise ValueError(
                "O servidor não retornou um arquivo ZIP válido."
            )

        temporario.replace(ZIP_LOCAL)

        with zipfile.ZipFile(ZIP_LOCAL) as pacote:
            candidatos = [
                item
                for item in pacote.infolist()
                if not item.is_dir()
                and PurePosixPath(item.filename).name == NOME_MAPA
                and "__MACOSX" not in item.filename
            ]

            if len(candidatos) != 1:
                raise ValueError(
                    f"Foram encontrados {len(candidatos)} mapas "
                    f"com o nome {NOME_MAPA}.\n"
                    "A seleção precisa ser conferida. "
                    "Os perfis celulares já importados continuam salvos."
                )

            # Ler somente o arquivo solicitado, sem extrair o pacote inteiro.
            CAMINHO_MAPA.write_bytes(
                pacote.read(candidatos[0])
            )

    except (requests.RequestException, zipfile.BadZipFile) as erro:
        raise RuntimeError(
            "Não foi possível obter o mapa experimental.\n"
            "O Parquet não normalizado permanece salvo.\n"
            f"Detalhe: {erro}"
        ) from erro

# Manter os campos como texto nesta primeira inspeção.
mapa_imtm_original = pl.read_csv(
    CAMINHO_MAPA,
    infer_schema=False,
    ignore_errors=False,
)

if mapa_imtm_original.is_empty():
    raise ValueError("O mapa experimental está vazio.")

manifesto_mapa = {
    "fonte": URL_ARQUIVO_AUTORES,
    "arquivo": NOME_MAPA,
    "sha256": hashlib.sha256(
        CAMINHO_MAPA.read_bytes()
    ).hexdigest(),
    "registros": mapa_imtm_original.height,
    "colunas": mapa_imtm_original.columns,
    "associacao_com_os_perfis_realizada": False,
}

(PASTA_MAPA / "manifesto_mapa.json").write_text(
    json.dumps(manifesto_mapa, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\nMAPA EXPERIMENTAL IMPORTADO")
print("=" * 65)
print(f"Registros: {mapa_imtm_original.height:,}")
print(f"Colunas: {mapa_imtm_original.width:,}")
print(f"Arquivo: {CAMINHO_MAPA}")

print("\nCOLUNAS DO MAPA")
for coluna in mapa_imtm_original.columns:
    print(coluna)

print("\nTabela disponível em: mapa_imtm_original")
print("Ainda não houve junção com os perfis ou normalização.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

import polars as pl
from IPython.display import display

# ============================================================
# 1. Conferir as entradas
# ============================================================

for nome in ["df_imtm_raw", "mapa_imtm_original", "PASTA_EXECUCAO"]:
    if nome not in globals():
        raise NameError(
            f"Variável ausente: {nome}. Execute as células anteriores."
        )

for tabela in (df_imtm_raw, mapa_imtm_original):
    if not isinstance(tabela, pl.DataFrame) or tabela.is_empty():
        raise TypeError(
            "As entradas precisam ser DataFrames Polars não vazios."
        )

K = [
    "Metadata_Plate_ID",
    "Metadata_Replicate",
    "Metadata_Well",
]

G = [
    "Metadata_Plate_ID",
    "Metadata_Replicate",
]

MAP_COLS = [
    "Metadata_Plate",
    "Metadata_Well",
    "Metadata_EOS",
    "Metadata_Concentration",
    "Metadata_Partner",
    "Metadata_Batch",
]

if (
    not set(K).issubset(df_imtm_raw.columns)
    or not set(MAP_COLS).issubset(mapa_imtm_original.columns)
):
    raise ValueError("Faltam chaves ou colunas do mapa experimental.")

# ============================================================
# 2. Preparar as chaves, preservando os metadados originais
# ============================================================

mapa = mapa_imtm_original.select(MAP_COLS).rename({
    "Metadata_Plate": "Metadata_Plate_Mapa",
    "Metadata_Batch": "Metadata_Batch_Mapa",
})

p = pl.col("Metadata_Plate_Mapa")
b = pl.col("Metadata_Batch_Mapa")

# Aceitar placa simples (B1001) ou acompanhada da réplica
# (B1001_R1). A réplica também pode estar em Metadata_Batch.
rep_placa = p.str.extract(r"_(R[1-4])$", 1)
rep_batch = b.str.extract(r"^(R[1-4])$", 1)

# Quando as duas informações existem, precisam concordar.
conflitos = mapa.filter(
    rep_placa.is_not_null()
    & rep_batch.is_not_null()
    & (rep_placa != rep_batch)
)

if conflitos.height:
    raise ValueError(
        "O mapa tem indicações conflitantes de réplica."
    )

mapa = mapa.with_columns(
    p.str.extract(
        r"^(B100[1-7])(?:_R[1-4])?$", 1
    ).alias("Metadata_Plate_ID"),

    pl.coalesce(
        [rep_placa, rep_batch]
    ).alias("Metadata_Replicate"),
)

# ============================================================
# 3. Verificar unicidade e correspondência das chaves
# ============================================================

for nome, tabela in [("perfis", df_imtm_raw), ("mapa", mapa)]:

    tem_null = tabela.select(
        pl.any_horizontal([
            pl.col(c).is_null() for c in K
        ]).any()
    ).item()

    tem_duplicacao = tabela.select(K).is_duplicated().any()

    if tem_null or tem_duplicacao:
        raise ValueError(
            f"Chave nula, formato não reconhecido ou duplicação em {nome}."
        )

if (
    set(mapa.select(G).unique().rows())
    != set(df_imtm_raw.select(G).unique().rows())
):
    raise ValueError(
        "Os grupos placa/réplica do mapa e dos perfis diferem."
    )

# Criar uma pasta nova para esta etapa.
pasta_base = Path(PASTA_EXECUCAO)

if not pasta_base.is_dir():
    raise FileNotFoundError(
        "A pasta desta execução não está acessível."
    )

PASTA_NORM = pasta_base / (
    "normalizacao_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
)

PASTA_NORM.mkdir(exist_ok=False)

# Anti-join: localizar as chaves que não aparecem na outra tabela.
pocos_sem_mapa = df_imtm_raw.select(K).join(
    mapa.select(K),
    on=K,
    how="anti",
)

pocos_sem_perfil = mapa.join(
    df_imtm_raw.select(K),
    on=K,
    how="anti",
)

pocos_sem_mapa.write_csv(
    PASTA_NORM / "pocos_sem_mapa.csv"
)

pocos_sem_perfil.write_csv(
    PASTA_NORM / "pocos_previstos_sem_perfil.csv"
)

if pocos_sem_mapa.height:
    raise ValueError(
        "Há perfis sem correspondência no mapa. "
        "Veja o arquivo pocos_sem_mapa.csv antes de continuar."
    )

# ============================================================
# 4. Associar o mapa, sem criar ou excluir perfis
# ============================================================

df_imtm_anotado = (
    df_imtm_raw
    .join(
        mapa,
        on=K,
        how="left",
        validate="1:1",
    )
    .sort(K)
)

if df_imtm_anotado.height != df_imtm_raw.height:
    raise RuntimeError("A junção alterou o número de perfis.")

sem_codigo = df_imtm_anotado.filter(
    pl.col("Metadata_EOS").is_null()
    | (pl.col("Metadata_EOS").str.strip_chars() == "")
)

if sem_codigo.height:
    raise ValueError(
        "Há perfis sem código experimental Metadata_EOS."
    )

# ============================================================
# 5. Conferir poços e controles por placa/réplica
# ============================================================

previstos = mapa.group_by(G).agg(
    pl.len().alias("n_previstos"),

    (pl.col("Metadata_EOS") == "DMSO")
    .sum()
    .alias("dmso_previstos"),
)

observados = df_imtm_anotado.group_by(G).agg(
    pl.len().alias("n_presentes"),

    (pl.col("Metadata_EOS") == "DMSO")
    .sum()
    .alias("dmso_presentes"),
)

conferencia_pocos = (
    previstos
    .join(observados, on=G, validate="1:1")
    .with_columns(
        (
            pl.col("n_previstos") - pl.col("n_presentes")
        ).alias("sem_perfil"),

        (
            pl.col("dmso_previstos") - pl.col("dmso_presentes")
        ).alias("dmso_sem_perfil"),
    )
    .sort(G)
)

conferencia_pocos.write_csv(
    PASTA_NORM / "conferencia_pocos.csv"
)

if conferencia_pocos.filter(
    pl.col("dmso_presentes") == 0
).height:
    raise ValueError(
        "Existe placa/réplica sem DMSO. Não normalize esse grupo."
    )

print("\nJUNÇÃO CONCLUÍDA")
print("=" * 65)
print(f"Perfis preservados: {df_imtm_anotado.height:,}")
print(f"Perfis sem mapa: {pocos_sem_mapa.height:,}")
print(f"Poços do mapa sem perfil: {pocos_sem_perfil.height:,}")

with pl.Config(tbl_rows=30, tbl_cols=8):
    display(conferencia_pocos)

In [ ]:
import numpy as np
import scipy

from scipy.stats import median_abs_deviation

# ============================================================
# 1. Parâmetros e medidas
# ============================================================

EPSILON = 1e-18
MAD_SCALE = 1 / 1.4826

features_norm = [
    c for c in df_imtm_anotado.columns
    if c.startswith(("Nuc_", "Cells_", "Cyto_"))
]

if not features_norm or any(
    not df_imtm_anotado.schema[c].is_numeric()
    for c in features_norm
):
    raise ValueError(
        "As medidas precisam existir e ser numéricas."
    )

metas_norm = [
    c for c in df_imtm_anotado.columns
    if c not in features_norm
]

# ============================================================
# 2. Cálculo numérico da transformação
# ============================================================

def calcular_mad_robusto(x, mascara_dmso):
    """
    Calcula a referência somente nos DMSO e transforma todo o grupo.

    Não imputa valores.
    Não seleciona medidas.
    Não modifica a matriz recebida.
    """

    if np.isinf(x).any():
        raise ValueError(
            "Há infinitos na entrada; revisar antes de normalizar."
        )

    controles = x[mascara_dmso]

    if controles.shape[0] == 0:
        raise ValueError("Nenhum DMSO disponível.")

    n_validos = np.sum(~np.isnan(controles), axis=0)

    mediana = np.full(x.shape[1], np.nan)
    mad = np.full(x.shape[1], np.nan)

    disponiveis = n_validos > 0

    if disponiveis.any():

        mediana[disponiveis] = np.nanmedian(
            controles[:, disponiveis],
            axis=0,
        )

        # SciPy divide a MAD pelo argumento scale.
        # Portanto, scale=1/1.4826 produz MAD × 1.4826.
        mad[disponiveis] = median_abs_deviation(
            controles[:, disponiveis],
            axis=0,
            nan_policy="omit",
            scale=MAD_SCALE,
        )

    with np.errstate(
        invalid="ignore",
        over="ignore",
        divide="ignore",
    ):
        z = (x - mediana) / (mad + EPSILON)

    return z, mediana, mad, n_validos

# ============================================================
# 3. Normalizar cada combinação placa/réplica
# ============================================================

partes_norm = []
partes_parametros = []
registros_norm = []

grupos = (
    df_imtm_anotado
    .select(G)
    .unique()
    .sort(G)
)

for i, (placa, replica) in enumerate(
    grupos.iter_rows(),
    start=1,
):

    bloco = df_imtm_anotado.filter(
        (pl.col(G[0]) == placa)
        & (pl.col(G[1]) == replica)
    )

    # Apenas as medidas entram no cálculo.
    # Os metadados, incluindo contagem celular, ficam intactos.
    x = (
        bloco
        .select(features_norm)
        .to_numpy()
        .astype(np.float64, copy=False)
    )

    dmso = (
        bloco["Metadata_EOS"] == "DMSO"
    ).to_numpy()

    z, mediana, mad, n_validos = calcular_mad_robusto(
        x,
        dmso,
    )

    # Representar resultados ausentes como null no Polars.
    # Isso não preenche os valores.
    numeros = (
        pl.from_numpy(z, schema=features_norm)
        .fill_nan(None)
    )

    partes_norm.append(
        bloco.with_columns(numeros.get_columns())
    )

    # Guardar os parâmetros de cada medida em cada grupo.
    parametros = (
        pl.DataFrame({
            "medida": features_norm,
            "mediana_dmso": mediana,
            "mad_escalado_dmso": mad,
            "n_dmso_validos": n_validos,
            "mad_zero": mad == 0,
            "sem_referencia_dmso": n_validos == 0,
            "menos_de_2_dmso_validos": n_validos < 2,
        })
        .with_columns(
            pl.lit(placa).alias(G[0]),
            pl.lit(replica).alias(G[1]),
            pl.col("mediana_dmso").fill_nan(None),
            pl.col("mad_escalado_dmso").fill_nan(None),
        )
    )

    partes_parametros.append(parametros)

    # Conferência numérica: mediana dos controles normalizados.
    # Usar somente medidas com MAD positiva e finita.
    regulares = np.isfinite(mad) & (mad > 0)

    erro_centro = None

    if regulares.any():
        centros = np.nanmedian(
            z[dmso][:, regulares],
            axis=0,
        )
        erro_centro = float(np.max(np.abs(centros)))

    registros_norm.append({
        G[0]: placa,
        G[1]: replica,
        "n_pocos": bloco.height,
        "n_dmso": int(dmso.sum()),
        "n_medidas_mad_zero": int((mad == 0).sum()),
        "n_medidas_sem_referencia": int((n_validos == 0).sum()),
        "n_ausentes_entrada": int(np.isnan(x).sum()),
        "n_ausentes_saida": int(np.isnan(z).sum()),
        "n_inf_saida": int(np.isinf(z).sum()),
        "max_abs_mediana_dmso_norm": erro_centro,
    })

    print(
        f"[{i:02d}/{grupos.height}] {placa} {replica}: "
        f"{int(dmso.sum())} DMSO; "
        f"{int((mad == 0).sum())} medidas com MAD zero",
        flush=True,
    )

# ============================================================
# 4. Reunir resultados e conferir preservação dos metadados
# ============================================================

df_imtm_norm = (
    pl.concat(partes_norm, how="vertical", rechunk=False)
    .sort(K)
)

parametros_normalizacao = pl.concat(
    partes_parametros,
    how="vertical",
)

relatorio_normalizacao = pl.DataFrame(registros_norm)

if (
    df_imtm_norm.shape != df_imtm_anotado.shape
    or not df_imtm_norm.select(metas_norm).equals(
        df_imtm_anotado.select(metas_norm)
    )
):
    raise RuntimeError(
        "As dimensões ou os metadados foram alterados."
    )

del partes_norm, partes_parametros

print("\nNORMALIZAÇÃO CONCLUÍDA")
print("=" * 65)
print(f"Linhas preservadas: {df_imtm_norm.height:,}")
print(f"Medidas normalizadas: {len(features_norm):,}")
print(f"Grupos placa/réplica: {grupos.height}")
print(
    "Valores ausentes na saída: "
    f"{relatorio_normalizacao['n_ausentes_saida'].sum():,}"
)
print(
    "Valores infinitos na saída: "
    f"{relatorio_normalizacao['n_inf_saida'].sum():,}"
)

with pl.Config(tbl_rows=30, tbl_cols=6):
    display(
        relatorio_normalizacao.select(
            G + [
                "n_pocos",
                "n_dmso",
                "n_medidas_mad_zero",
                "n_medidas_sem_referencia",
            ]
        )
    )

print("\nTabela normalizada: df_imtm_norm")
print("Parâmetros por medida/grupo: parametros_normalizacao")
print("Conferências por grupo: relatorio_normalizacao")
print("Nenhuma medida ou réplica foi excluída.")

In [ ]:
import json
import sys

ARQUIVO_NORM = (
    PASTA_NORM / "imtm_hepg2_mad_robustize.parquet"
)

df_imtm_norm.write_parquet(
    ARQUIVO_NORM,
    compression="zstd",
)

parametros_normalizacao.write_parquet(
    PASTA_NORM / "parametros_normalizacao.parquet",
    compression="zstd",
)

relatorio_normalizacao.write_csv(
    PASTA_NORM / "relatorio_normalizacao.csv"
)

# Conferir dimensões e esquema do arquivo salvo.
verificacao = pl.scan_parquet(ARQUIVO_NORM)

if (
    verificacao.select(pl.len()).collect().item()
    != df_imtm_norm.height
    or list(verificacao.collect_schema().items())
    != list(df_imtm_norm.schema.items())
):
    raise RuntimeError(
        "A estrutura do Parquet salvo difere do resultado."
    )

manifesto = {
    "metodo": "mad_robustize",
    "implementacao": "NumPy/SciPy; tabelas em Polars",
    "grupos": G,
    "controle": "Metadata_EOS == 'DMSO'",
    "formula": (
        "(x - mediana_DMSO) / (MAD_escalado_DMSO + epsilon)"
    ),
    "scipy_mad_scale": MAD_SCALE,
    "epsilon": EPSILON,
    "features": features_norm,
    "n_perfis": df_imtm_norm.height,
    "n_pocos_mapa_sem_perfil": pocos_sem_perfil.height,
    "imputacao_aplicada": False,
    "selecao_de_features_aplicada": False,
    "consenso_entre_replicas_aplicado": False,
    "comparacao_com_arquivo_dos_autores_realizada": False,
    "nan_representado_como_null_na_saida": True,
    "python": sys.version.split()[0],
    "polars": pl.__version__,
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "fontes": [
        (
            "https://github.com/schmiedc/EU-OS_bioactives/"
            "blob/main/Analysis_IMTM/"
            "2_Normalization_IMTM_HepG2.ipynb"
        ),
        (
            "https://github.com/cytomining/pycytominer/"
            "blob/main/pycytominer/operations/transform.py"
        ),
    ],
    "resumo_por_grupo": relatorio_normalizacao.to_dicts(),
}

(PASTA_NORM / "manifesto_normalizacao.json").write_text(
    json.dumps(manifesto, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\nRESULTADOS SALVOS")
print("=" * 65)
print(f"Parquet normalizado: {ARQUIVO_NORM}")
print(f"Pasta dos relatórios: {PASTA_NORM}")
print("\nOs arquivos de entrada permanecem intactos.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict

import json
import hashlib
import numpy as np
import scipy
from scipy.stats import rankdata
import polars as pl

from IPython.display import display

# ============================================================
# 1. Arquivos e parâmetros da seleção
# ============================================================

PASTA_NORM = Path(
    "/content/drive/MyDrive/mba_tcc/pipeline_IMTM_HepG2/"
    "20260919T204105_625149Z/"
    "normalizacao_20260919T205516_579360Z"
)

ARQUIVO_NORM = PASTA_NORM / "imtm_hepg2_mad_robustize.parquet"
ARQUIVO_PARAM = PASTA_NORM / "parametros_normalizacao.parquet"

REPS_DESENV = ["R1", "R2", "R3"]
REP_CONFIRMACAO = "R4"

LIMIAR_RHO = 0.90
MIN_DMSO = 10
MIN_MAD = 1e-18

G = ["Metadata_Plate_ID", "Metadata_Replicate"]
K = G + ["Metadata_Well"]

for caminho in [ARQUIVO_NORM, ARQUIVO_PARAM]:
    if not caminho.is_file():
        raise FileNotFoundError(
            f"Arquivo não encontrado:\n{caminho}"
        )

base_fs = pl.read_parquet(ARQUIVO_NORM).sort(K)
param_fs = pl.read_parquet(ARQUIVO_PARAM)

features_fs_entrada = [
    c for c in base_fs.columns
    if c.startswith(("Nuc_", "Cells_", "Cyto_"))
]

metas_fs = [
    c for c in base_fs.columns
    if c not in features_fs_entrada
]

# ============================================================
# 2. Conferir estrutura e identificação dos poços
# ============================================================

if not features_fs_entrada or any(
    not base_fs.schema[c].is_numeric()
    for c in features_fs_entrada
):
    raise ValueError("Medidas ausentes ou não numéricas.")

if base_fs.is_empty() or base_fs.select(K).is_duplicated().any():
    raise ValueError("Base vazia ou chaves de poços repetidas.")

if base_fs.select(
    pl.any_horizontal([
        pl.col(c).is_null() for c in K
    ]).any()
).item():
    raise ValueError("Há chaves nulas.")

if set(base_fs[G[1]].unique()) != set(
    REPS_DESENV + [REP_CONFIRMACAO]
):
    raise ValueError(
        "As réplicas não correspondem a R1, R2, R3 e R4."
    )

grupos_fs = base_fs.select(G).unique().sort(G)

# Os parâmetros devem cobrir exatamente todas as medidas
# em todas as combinações de placa e réplica.
if (
    param_fs.select(G + ["medida"]).is_duplicated().any()
    or set(param_fs["medida"].unique()) != set(features_fs_entrada)
    or set(param_fs.select(G).unique().rows()) != set(grupos_fs.rows())
    or param_fs.height != grupos_fs.height * len(features_fs_entrada)
):
    raise ValueError(
        "Os parâmetros não cobrem exatamente todas as medidas e grupos."
    )

# ============================================================
# 3. Identificar escalas de normalização utilizáveis
# ============================================================

param_fs = param_fs.with_columns(
    (
        pl.col("mediana_dmso").is_finite()
        & pl.col("mad_escalado_dmso").is_finite()
        & (pl.col("mad_escalado_dmso") > MIN_MAD)
        & (pl.col("n_dmso_validos") >= MIN_DMSO)
    )
    .fill_null(False)
    .alias("escala_utilizavel")
)

# SOMENTE R1-R3 entram na escolha das variáveis.
desenvolvimento_fs = base_fs.filter(
    pl.col(G[1]).is_in(REPS_DESENV)
)

qualidade_dev = (
    param_fs
    .filter(pl.col(G[1]).is_in(REPS_DESENV))
    .group_by("medida")
    .agg(
        (~pl.col("escala_utilizavel"))
        .sum()
        .alias("n_grupos_escala_invalida_dev")
    )
)

problemas_escala = dict(qualidade_dev.iter_rows())

print("ENTRADA PREPARADA")
print("=" * 65)
print(f"Poços na base: {base_fs.height:,}")
print(f"Medidas de entrada: {len(features_fs_entrada):,}")
print(f"Poços para selecionar variáveis: {desenvolvimento_fs.height:,}")
print("R4 não participa da escolha das variáveis.")

In [ ]:
# ============================================================
# 1. Classificar cada medida pelo canal e pela família
# ============================================================

# Um canal só será criado se estiver presente nos nomes.
CANAIS = {
    "mito": "Mito",
    "dna": "DNA",
    "er": "ER",
    "agp": "AGP",
    "rna": "RNA",
}


def classificar_medida(nome):
    objeto, familia, metrica = nome.split("_", 2)

    canais = sorted({
        CANAIS[token]
        for token in metrica.lower().split("_")
        if token in CANAIS
    })

    tecnico = (
        familia in {"Location", "ImageQuality", "Threshold"}
        or (
            familia == "AreaShape"
            and metrica.startswith((
                "Center_",
                "BoundingBoxMaximum_",
                "BoundingBoxMinimum_",
            ))
        )
        or (
            familia == "Neighbors"
            and "ObjectNumber" in metrica
        )
    )

    if tecnico:
        return familia, "TECNICO", "tecnico"

    if len(canais) > 1:
        return (
            familia,
            "Intercanais_" + "_".join(canais),
            "complementar",
        )

    if len(canais) == 1:
        return familia, canais[0], "principal"

    # Forma e vizinhança sem canal explícito:
    # preservar separadamente por objeto medido.
    if familia in {"AreaShape", "Neighbors"}:
        return familia, "Morfologia_" + objeto, "principal"

    # Não atribuir um canal arbitrariamente.
    return familia, "REVISAR", "revisar"


# ============================================================
# 2. Aplicar os filtros de qualidade usando somente R1-R3
# ============================================================

registro_fs = {}

print("Conferindo as medidas em R1-R3...", flush=True)

for i, coluna in enumerate(features_fs_entrada, start=1):
    familia, bloco, uso = classificar_medida(coluna)

    valores = (
        desenvolvimento_fs[coluna]
        .to_numpy()
        .astype(np.float64, copy=False)
    )

    finitos = valores[np.isfinite(valores)]

    n_invalidos = int(valores.size - finitos.size)
    n_distintos = int(np.unique(finitos).size)

    status = "elegivel"

    if uso in {"tecnico", "revisar"}:
        status = "reservada_" + uso

    elif problemas_escala[coluna] > 0:
        status = "excluida_escala_dmso"

    elif n_invalidos > 0:
        status = "excluida_ausente_ou_nao_finita"

    elif n_distintos <= 1:
        status = "excluida_constante"

    registro_fs[coluna] = {
        "medida": coluna,
        "familia": familia,
        "bloco": bloco,
        "uso": uso,
        "n_grupos_escala_invalida_dev": problemas_escala[coluna],
        "n_invalidos_dev": n_invalidos,
        "n_distintos_dev": n_distintos,

        # Informação para revisão, não um critério de exclusão.
        "max_abs_dev": (
            float(np.max(np.abs(finitos)))
            if finitos.size else None
        ),

        "status": status,
        "representante": None,
        "abs_rho": None,
    }

    if i % 500 == 0 or i == len(features_fs_entrada):
        print(
            f"  {i}/{len(features_fs_entrada)} medidas verificadas.",
            flush=True,
        )

elegiveis_fs = [
    c for c in features_fs_entrada
    if registro_fs[c]["status"] == "elegivel"
]

# ============================================================
# 3. Reduzir redundâncias DENTRO de cada bloco e família
# ============================================================

subgrupos_fs = defaultdict(list)

for coluna in elegiveis_fs:
    registro = registro_fs[coluna]

    subgrupos_fs[
        (registro["bloco"], registro["familia"])
    ].append(coluna)

for (bloco, familia), nomes in sorted(subgrupos_fs.items()):
    nomes = sorted(nomes)

    if len(nomes) == 1:
        registro_fs[nomes[0]]["status"] = "selecionada"
        continue

    matriz = desenvolvimento_fs.select(nomes).to_numpy()

    postos = rankdata(
        matriz,
        axis=0,
        method="average",
    )

    rho = np.corrcoef(postos, rowvar=False)

    if not np.isfinite(rho).all():
        raise RuntimeError(
            f"Correlação indefinida em {bloco}/{familia}."
        )

    mantidos = []

    for j, coluna in enumerate(nomes):
        if mantidos:
            posicao = int(
                np.argmax(np.abs(rho[j, mantidos]))
            )
            k = mantidos[posicao]
            valor = float(abs(rho[j, k]))

            if valor >= LIMIAR_RHO:
                registro_fs[coluna].update({
                    "status": "excluida_redundancia",
                    "representante": nomes[k],
                    "abs_rho": valor,
                })
                continue

        mantidos.append(j)
        registro_fs[coluna]["status"] = "selecionada"

    print(
        f"{bloco}/{familia}: "
        f"{len(nomes)} -> {len(mantidos)}",
        flush=True,
    )

# ============================================================
# 4. Consolidar a seleção e mostrar a cobertura dos blocos
# ============================================================

catalogo_fs = pl.DataFrame(
    list(registro_fs.values()),
    infer_schema_length=None,
    schema_overrides={
        "representante": pl.String,
        "abs_rho": pl.Float64,
    },
)

FEATURES_FS = [
    c for c in features_fs_entrada
    if registro_fs[c]["status"] == "selecionada"
]

if not FEATURES_FS:
    raise RuntimeError(
        "Nenhuma medida selecionada. Consulte catalogo_fs."
    )

FEATURES_PRINCIPAIS = [
    c for c in FEATURES_FS
    if registro_fs[c]["uso"] == "principal"
]

FEATURES_COMPLEMENTARES = [
    c for c in FEATURES_FS
    if registro_fs[c]["uso"] == "complementar"
]

FEATURES_POR_BLOCO = {
    bloco: [
        c for c in FEATURES_FS
        if registro_fs[c]["bloco"] == bloco
    ]
    for bloco in sorted({
        registro_fs[c]["bloco"] for c in FEATURES_FS
    })
}

resumo_fs = (
    catalogo_fs
    .group_by("bloco", "uso")
    .agg(
        pl.len().alias("entrada"),

        pl.col("status")
        .is_in(["selecionada", "excluida_redundancia"])
        .sum()
        .alias("apos_qualidade"),

        (pl.col("status") == "selecionada")
        .sum()
        .alias("selecionadas"),
    )
    .sort("uso", "bloco")
)

print("\nCOBERTURA APÓS A SELEÇÃO")

with pl.Config(tbl_rows=30, tbl_cols=6):
    display(resumo_fs)

for linha in resumo_fs.iter_rows(named=True):
    if linha["uso"] == "principal" and linha["selecionadas"] == 0:
        print(
            f"ATENÇÃO: {linha['bloco']} ficou sem medidas selecionadas. "
            "Esse bloco não poderá ser considerado preservado."
        )

In [ ]:
# ============================================================
# 1. Identificar bloqueios técnicos na aplicação
# ============================================================

bloqueios_aplicacao = param_fs.filter(
    pl.col("medida").is_in(FEATURES_FS)
    & ~pl.col("escala_utilizavel")
)

bloqueios_por_grupo = defaultdict(list)

for placa, replica, medida in (
    bloqueios_aplicacao
    .select(G + ["medida"])
    .iter_rows()
):
    bloqueios_por_grupo[(placa, replica)].append(medida)

# ============================================================
# 2. Criar a tabela selecionada, preservando todos os poços
# ============================================================

partes_fs = []

base_selecionada = base_fs.select(
    metas_fs + FEATURES_FS
)

for placa, replica in grupos_fs.iter_rows():
    bloco = base_selecionada.filter(
        (pl.col(G[0]) == placa)
        & (pl.col(G[1]) == replica)
    )

    # Representar valores não finitos como ausências.
    # Não preencher nenhuma ausência.
    bloco = bloco.with_columns([
        pl.when(pl.col(c).is_finite())
        .then(pl.col(c))
        .otherwise(None)
        .alias(c)
        for c in FEATURES_FS
    ])

    # Bloquear a medida somente no grupo sem escala utilizável.
    bloco = bloco.with_columns([
        pl.lit(None, dtype=base_fs.schema[c]).alias(c)
        for c in bloqueios_por_grupo[(placa, replica)]
    ])

    partes_fs.append(bloco)

df_imtm_fs = (
    pl.concat(
        partes_fs,
        how="vertical",
        rechunk=False,
    )
    .sort(K)
)

del partes_fs

if (
    df_imtm_fs.height != base_fs.height
    or not df_imtm_fs.select(metas_fs).equals(
        base_fs.select(metas_fs)
    )
):
    raise RuntimeError(
        "O número de poços ou os metadados foram alterados."
    )

df_imtm_fs_dev = df_imtm_fs.filter(
    pl.col(G[1]).is_in(REPS_DESENV)
)

df_imtm_fs_confirmacao = df_imtm_fs.filter(
    pl.col(G[1]) == REP_CONFIRMACAO
)

# ============================================================
# 3. Salvar em uma pasta nova
# ============================================================

PASTA_FS = PASTA_NORM / (
    "feature_selection_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
)

PASTA_FS.mkdir(exist_ok=False)

ARQUIVO_FS = (
    PASTA_FS / "imtm_hepg2_features_selecionadas.parquet"
)

df_imtm_fs.write_parquet(
    ARQUIVO_FS,
    compression="zstd",
)

catalogo_fs.write_csv(
    PASTA_FS / "catalogo_selecao.csv"
)

resumo_fs.write_csv(
    PASTA_FS / "resumo_por_bloco.csv"
)

bloqueios_aplicacao.write_csv(
    PASTA_FS / "bloqueios_escala_aplicacao.csv"
)

(PASTA_FS / "features_por_bloco.json").write_text(
    json.dumps(
        FEATURES_POR_BLOCO,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


def sha256_arquivo(caminho):
    resumo_hash = hashlib.sha256()

    with caminho.open("rb") as arquivo:
        for parte in iter(
            lambda: arquivo.read(1024 * 1024),
            b"",
        ):
            resumo_hash.update(parte)

    return resumo_hash.hexdigest()


manifesto_fs = {
    "entrada": str(ARQUIVO_NORM),
    "sha256_entrada": sha256_arquivo(ARQUIVO_NORM),
    "parametros": str(ARQUIVO_PARAM),
    "sha256_parametros": sha256_arquivo(ARQUIVO_PARAM),
    "tipo": "selecao_adaptada_para_seletividade_multicanal",
    "replicas_ajuste": REPS_DESENV,
    "replica_confirmacao": REP_CONFIRMACAO,
    "metodo_correlacao": "spearman",
    "limiar_abs_rho": LIMIAR_RHO,
    "grupos_correlacao": ["bloco", "familia"],
    "ordem_representantes": "alfabetica",
    "min_dmso_validos": MIN_DMSO,
    "mad_minimo_exclusivo": MIN_MAD,
    "criterio_nao_finitos": "nenhum permitido em R1-R3",
    "criterio_constante": "no maximo um valor distinto em R1-R3",
    "corte_de_outliers_por_magnitude": None,
    "imputacao": False,
    "comparacao_autores": "pendente",
    "features_antes_correlacao": elegiveis_fs,
    "features_selecionadas": FEATURES_FS,
    "features_principais": FEATURES_PRINCIPAIS,
    "features_complementares": FEATURES_COMPLEMENTARES,
    "n_linhas": df_imtm_fs.height,
    "n_bloqueios_medida_grupo": bloqueios_aplicacao.height,
    "versoes": {
        "polars": pl.__version__,
        "numpy": np.__version__,
        "scipy": scipy.__version__,
    },
}

(PASTA_FS / "manifesto_selecao.json").write_text(
    json.dumps(
        manifesto_fs,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

# ============================================================
# 4. Conferir a estrutura do arquivo salvo
# ============================================================

conferencia = pl.scan_parquet(ARQUIVO_FS)

if (
    conferencia.select(pl.len()).collect().item()
    != df_imtm_fs.height
    or list(conferencia.collect_schema().items())
    != list(df_imtm_fs.schema.items())
):
    raise RuntimeError(
        "Falha na verificação do arquivo salvo."
    )

print("\nSELEÇÃO CONCLUÍDA")
print("=" * 65)
print(f"Poços preservados: {df_imtm_fs.height:,}")
print(f"Medidas de entrada: {len(features_fs_entrada):,}")
print(f"Após filtros de qualidade: {len(elegiveis_fs):,}")
print(f"Após filtro de redundância: {len(FEATURES_FS):,}")
print(f"Medidas principais: {len(FEATURES_PRINCIPAIS):,}")
print(f"Medidas complementares: {len(FEATURES_COMPLEMENTARES):,}")
print(f"Poços de desenvolvimento: {df_imtm_fs_dev.height:,}")
print(f"Poços reservados em R4: {df_imtm_fs_confirmacao.height:,}")
print(
    "Pares medida/grupo bloqueados na aplicação: "
    f"{bloqueios_aplicacao.height:,}"
)
print(f"Resultados: {PASTA_FS}")
print("\nO arquivo normalizado original permanece intacto.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

import json
import hashlib
import numpy as np
import polars as pl

from IPython.display import display

PASTA_FS = Path(
    "/content/drive/MyDrive/mba_tcc/pipeline_IMTM_HepG2/"
    "20260919T204105_625149Z/"
    "normalizacao_20260919T205516_579360Z/"
    "feature_selection_20260919T211235_656816Z"
)

P = "Metadata_Plate_ID"
R = "Metadata_Replicate"
EOS = "Metadata_EOS"
DOSE = "Metadata_Concentration"
COUNT = "Metadata_Object_Count"

KEY = [P, R, "Metadata_Well"]
TREAT = [P, EOS, DOSE]

REPS = ["R1", "R2", "R3"]
QUANTIL = 0.95
MIN_DMSO_POR_REPLICA = 10


def rms_completo(x):
    """RMS com cobertura completa; não trata ausências como zero."""

    out = np.full(x.shape[0], np.nan)
    valid = np.isfinite(x).all(axis=1)

    # Escalonamento interno evita overflow ao elevar ao quadrado.
    a = np.abs(x[valid])

    if len(a):
        scale = a.max(axis=1)
        nonzero = scale > 0
        value = np.zeros(len(a))

        value[nonzero] = scale[nonzero] * np.sqrt(
            np.mean(
                (a[nonzero] / scale[nonzero, None]) ** 2,
                axis=1,
            )
        )

        out[valid] = value

    return out


def cosseno_minimo(x):
    """Menor cosseno entre réplicas; None quando não avaliável."""

    if len(x) < 2 or not np.isfinite(x).all():
        return None

    scale = np.abs(x).max(axis=1)

    if np.any(scale == 0):
        return None

    y = x / scale[:, None]
    y = y / np.linalg.norm(y, axis=1)[:, None]

    sim = np.clip(y @ y.T, -1.0, 1.0)

    return float(
        sim[np.triu_indices(len(x), 1)].min()
    )


def calcular_escores(pasta):
    arquivo = pasta / "imtm_hepg2_features_selecionadas.parquet"
    cat_path = pasta / "catalogo_selecao.csv"

    if not arquivo.is_file() or not cat_path.is_file():
        raise FileNotFoundError(
            "Confira a pasta da seleção e seus arquivos."
        )

    cat = pl.read_csv(cat_path).filter(
        pl.col("status") == "selecionada"
    )

    cols = cat["medida"].to_list()

    if not cols or len(cols) != len(set(cols)):
        raise ValueError(
            "Catálogo vazio ou com medidas repetidas."
        )

    # R4 fica fora dos cálculos e da calibração.
    dev = (
        pl.scan_parquet(arquivo)
        .filter(pl.col(R).is_in(REPS))
        .collect()
        .sort(KEY)
    )

    obrigatorias = set(KEY + [EOS, DOSE, COUNT] + cols)

    if dev.is_empty() or not obrigatorias.issubset(dev.columns):
        raise ValueError(
            "Dados ou metadados necessários ausentes."
        )

    if dev.select(KEY).is_duplicated().any():
        raise ValueError("Há poços duplicados.")

    if dev.select(
        pl.any_horizontal([
            pl.col(c).is_null()
            for c in set(KEY + TREAT)
        ]).any()
    ).item():
        raise ValueError(
            "Chaves ou identificadores experimentais ausentes."
        )

    if any(not dev.schema[c].is_numeric() for c in cols):
        raise TypeError(
            "O catálogo deve conter apenas medidas numéricas."
        )

    # Reutilizar a classificação já salva.
    blocos = {
        b: cat.filter(
            pl.col("bloco") == b
        )["medida"].to_list()
        for b in sorted(cat["bloco"].unique().to_list())
    }

    principais = sorted(
        cat.filter(
            pl.col("uso") == "principal"
        )["bloco"].unique().to_list()
    )

    canais = [
        b for b in principais
        if b in ["Mito", "DNA", "ER", "RNA", "AGP"]
    ]

    morfologia = [
        b for b in principais
        if b.startswith("Morfologia_")
    ]

    if (
        not {"Mito", "DNA", "ER", "AGP"}.issubset(canais)
        or not morfologia
    ):
        raise ValueError(
            "Faltam blocos principais para avaliar o IMTM."
        )

    outros = [b for b in canais if b != "Mito"]

    metadados = [
        c for c in dev.columns
        if c.startswith("Metadata_")
    ]

    partes = []
    referencias = []

    # --------------------------------------------------------
    # RMS e referência DMSO por placa, reunindo R1-R3
    # --------------------------------------------------------

    for plate in dev.partition_by(P, maintain_order=True):
        placa = plate[P][0]
        dmso = (plate[EOS] == "DMSO").to_numpy()

        for rep in REPS:
            n = int(np.sum(
                dmso & (plate[R].to_numpy() == rep)
            ))

            if n < MIN_DMSO_POR_REPLICA:
                raise ValueError(
                    f"{placa}/{rep}: apenas {n} DMSO."
                )

        score = plate.select(metadados)
        novas = []

        for bloco, medidas in blocos.items():
            x = (
                plate.select(medidas)
                .to_numpy()
                .astype(float, copy=False)
            )

            e = rms_completo(x)
            ref = e[dmso & np.isfinite(e)]

            limite = (
                float(np.quantile(
                    ref, QUANTIL, method="linear"
                ))
                if len(ref) else np.nan
            )

            utilizavel = (
                len(ref) == int(dmso.sum())
                and np.isfinite(limite)
                and limite > 0
            )

            a = (
                e / limite
                if utilizavel
                else np.full(len(e), np.nan)
            )

            referencias.append({
                P: placa,
                "bloco": bloco,
                "n_medidas": len(medidas),
                "n_dmso_validos": len(ref),
                "quantil": QUANTIL,
                "limiar_RMS": limite,
                "referencia_utilizavel": bool(utilizavel),
            })

            novas.extend([
                pl.Series(f"E_{bloco}", e).fill_nan(None),
                pl.Series(f"A_{bloco}", a).fill_nan(None),
            ])

        partes.append(score.with_columns(novas))

        print(
            f"{placa}: escores calculados; "
            f"{int(dmso.sum())} DMSO em R1-R3.",
            flush=True,
        )

    scores = pl.concat(partes, how="vertical").sort(KEY)

    ref_df = pl.DataFrame(referencias).with_columns(
        pl.col("limiar_RMS").fill_nan(None)
    )

    # --------------------------------------------------------
    # Contagem celular relativa: informação de apoio
    # --------------------------------------------------------

    count_ref = (
        dev.filter(pl.col(EOS) == "DMSO")
        .group_by(P, R)
        .agg(
            pl.col(COUNT)
            .median()
            .alias("_contagem_dmso")
        )
    )

    scores = (
        scores.join(
            count_ref,
            on=[P, R],
            how="left",
            validate="m:1",
        )
        .with_columns(
            pl.when(
                (pl.col("_contagem_dmso") > 0)
                & pl.col(COUNT).is_finite()
            )
            .then(
                pl.col(COUNT) / pl.col("_contagem_dmso")
            )
            .otherwise(None)
            .alias("contagem_relativa_DMSO")
        )
        .drop("_contagem_dmso")
    )

    acol = [f"A_{b}" for b in principais]

    scores = scores.with_columns(
        pl.all_horizontal([
            pl.col(c).is_finite().fill_null(False)
            for c in acol
        ]).alias("avaliavel")
    )

    # --------------------------------------------------------
    # Uma observação por condição e réplica
    # --------------------------------------------------------

    rep = (
        scores.filter(pl.col(EOS) != "DMSO")
        .group_by(TREAT + [R])
        .agg(
            pl.len().alias("n_pocos"),
            pl.col("avaliavel").all(),
            *[pl.col(c).median() for c in acol],
            pl.col("contagem_relativa_DMSO").median(),
        )
        .with_columns(
            pl.when(pl.col("avaliavel"))
            .then(
                pl.max_horizontal([
                    f"A_{b}" for b in outros
                ])
            )
            .otherwise(None)
            .alias("A_outros_max"),

            pl.when(pl.col("avaliavel"))
            .then(
                pl.max_horizontal([
                    f"A_{b}" for b in morfologia
                ])
            )
            .otherwise(None)
            .alias("A_morfologia_max"),
        )
        .with_columns(
            (
                pl.col("avaliavel")
                & (pl.col("A_Mito") > 1)
            )
            .fill_null(False)
            .alias("mito_acima"),

            pl.when(pl.col("avaliavel"))
            .then(
                pl.col("A_Mito")
                / pl.max_horizontal(
                    pl.lit(1.0),
                    pl.col("A_outros_max"),
                    pl.col("A_morfologia_max"),
                )
            )
            .otherwise(None)
            .alias("contraste_mito"),
        )
    )

    summary = rep.group_by(TREAT).agg(
        pl.len().alias("n_replicas"),

        pl.col("avaliavel")
        .sum()
        .alias("n_replicas_avaliaveis"),

        pl.col("mito_acima")
        .sum()
        .alias("n_replicas_mito_acima"),

        pl.col("A_Mito")
        .median()
        .alias("A_Mito_mediana"),

        pl.col("A_outros_max")
        .max()
        .alias("A_outros_pior_replica"),

        pl.col("A_morfologia_max")
        .max()
        .alias("A_morfologia_pior_replica"),

        pl.col("contraste_mito")
        .median()
        .alias("contraste_mediano"),

        pl.col("contagem_relativa_DMSO")
        .min()
        .alias("contagem_relativa_min"),
    )

    # --------------------------------------------------------
    # Conferir a direção dos perfis mitocondriais
    # --------------------------------------------------------

    mito = blocos["Mito"]

    vetores = (
        dev.filter(pl.col(EOS) != "DMSO")
        .group_by(TREAT + [R])
        .agg(*[pl.col(c).median() for c in mito])
    )

    coerencia = []

    for grupo in vetores.partition_by(
        TREAT, maintain_order=True
    ):
        item = {k: grupo[k][0] for k in TREAT}

        item["cosseno_mito_min"] = cosseno_minimo(
            grupo.select(mito).to_numpy().astype(float)
        )

        coerencia.append(item)

    coerencia = pl.DataFrame(
        coerencia,
        schema_overrides={
            "cosseno_mito_min": pl.Float64
        },
    )

    summary = (
        summary.join(
            coerencia,
            on=TREAT,
            how="left",
            validate="1:1",
        )
        .with_columns(
            (
                (pl.col("n_replicas") == 3)
                & (pl.col("n_replicas_avaliaveis") == 3)
                & (pl.col("n_replicas_mito_acima") >= 2)
                & (pl.col("A_outros_pior_replica") <= 1)
                & (pl.col("A_morfologia_pior_replica") <= 1)
                & (pl.col("cosseno_mito_min") > 0)
            )
            .fill_null(False)
            .alias("prioridade_exploratoria")
        )
        .sort(
            ["prioridade_exploratoria", "contraste_mediano"],
            descending=[True, True],
            nulls_last=True,
        )
    )

    return dev, cat, scores, ref_df, rep, summary

In [ ]:
(
    perfis_dev,
    catalogo_escores,
    escores_pocos,
    referencias_dmso,
    escores_replicas,
    resumo_tratamentos,
) = calcular_escores(PASTA_FS)

prioridades = resumo_tratamentos.filter(
    pl.col("prioridade_exploratoria")
)

PASTA_ESCORES = PASTA_FS / (
    "escores_exploratorios_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
)

PASTA_ESCORES.mkdir(exist_ok=False)

escores_pocos.write_parquet(
    PASTA_ESCORES / "escores_pocos_R1_R3.parquet"
)

escores_replicas.write_parquet(
    PASTA_ESCORES / "escores_condicao_replica.parquet"
)

resumo_tratamentos.write_csv(
    PASTA_ESCORES / "resumo_tratamentos.csv"
)

referencias_dmso.write_csv(
    PASTA_ESCORES / "referencias_dmso.csv"
)

arquivo_entrada = (
    PASTA_FS / "imtm_hepg2_features_selecionadas.parquet"
)

hash_entrada = hashlib.sha256()

with arquivo_entrada.open("rb") as arquivo:
    for parte in iter(
        lambda: arquivo.read(1024 * 1024), b""
    ):
        hash_entrada.update(parte)

manifesto = {
    "arquivo_entrada": str(arquivo_entrada),
    "sha256_entrada": hash_entrada.hexdigest(),
    "replicas_analisadas": REPS,
    "R4_analisada": False,
    "escore": "RMS das medidas normalizadas",
    "referencia": "DMSO por placa, reunindo R1-R3",
    "quantil_referencia": QUANTIL,
    "metodo_quantil": "linear",
    "min_dmso_por_replica": MIN_DMSO_POR_REPLICA,
    "condicao": TREAT,
    "criterio_prioridade": {
        "replicas_avaliaveis": 3,
        "replicas_Mito_acima_referencia": "pelo menos 2",
        "outros_canais": "nenhuma replica acima da referencia",
        "morfologia": "nenhuma replica acima da referencia",
        "cosseno_Mito_minimo": "> 0",
    },
    "contagem_celular": "diagnostico; nao usada como filtro",
    "teste_de_equivalencia_realizado": False,
    "controle_FDR_realizado": False,
    "rotulos_biologicos_validados": False,
    "n_prioridades_exploratorias": prioridades.height,
    "polars": pl.__version__,
    "numpy": np.__version__,
}

(PASTA_ESCORES / "manifesto_escores.json").write_text(
    json.dumps(
        manifesto,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    ),
    encoding="utf-8",
)

print("\nESCORES EXPLORATÓRIOS CONCLUÍDOS")
print("=" * 65)
print(f"Poços analisados em R1-R3: {perfis_dev.height:,}")
print(f"Condições experimentais: {resumo_tratamentos.height:,}")
print(f"Prioridades exploratórias: {prioridades.height:,}")
print(
    "Referências não utilizáveis: "
    f"{referencias_dmso.filter(~pl.col('referencia_utilizavel')).height:,}"
)
print("R4 permanece reservada.")
print(f"Resultados: {PASTA_ESCORES}")

colunas_exibir = TREAT + [
    "n_replicas",
    "n_replicas_mito_acima",
    "A_Mito_mediana",
    "A_outros_pior_replica",
    "A_morfologia_pior_replica",
    "cosseno_mito_min",
    "contagem_relativa_min",
    "contraste_mediano",
]

with pl.Config(tbl_rows=20, tbl_cols=12, fmt_str_lengths=25):
    if prioridades.height:
        print("\nPRIORIDADES PARA INSPEÇÃO — NÃO SÃO HITS CONFIRMADOS")
        display(prioridades.select(colunas_exibir).head(20))
    else:
        print("\nNenhuma condição passou por todos os critérios.")
        print("Maiores contrastes, apresentados apenas para diagnóstico:")
        display(
            resumo_tratamentos
            .select(colunas_exibir)
            .head(20)
        )

print("\nNenhum arquivo de entrada foi alterado.")

In [ ]:
def explicar_condicao(
    dev,
    cat,
    placa,
    codigo,
    concentracao,
    top_k=10,
):
    cols = (
        cat.filter(pl.col("bloco") == "Mito")
        ["medida"].to_list()
    )

    selected = dev.filter(
        (pl.col(P) == placa)
        & (pl.col(EOS) == codigo)
        & (pl.col(DOSE) == concentracao)
    )

    v = (
        selected
        .group_by(R)
        .agg(*[pl.col(c).median() for c in cols])
        .sort(R)
    )

    if set(v[R].to_list()) != set(REPS):
        raise ValueError(
            "A explicação exige as três réplicas R1-R3."
        )

    x = v.select(cols).to_numpy().astype(float)

    if not np.isfinite(x).all():
        raise ValueError(
            "Há medidas ausentes ou não finitas nesta condição."
        )

    # A mesma escala em todos os elementos cancela no quociente.
    scale = np.abs(x).max()

    if scale == 0:
        raise ValueError(
            "O perfil é inteiramente zero; não há contribuição a decompor."
        )

    q = np.mean((x / scale) ** 2, axis=0)
    contribution = 100 * q / q.sum()

    values = {
        "medida": cols,
        "familia": [c.split("_")[1] for c in cols],
        "z_mediano": np.median(x, axis=0),
        "contribuicao_pct": contribution,
    }

    for i, replica in enumerate(v[R].to_list()):
        values[f"z_{replica}"] = x[i]

    tabela = (
        pl.DataFrame(values)
        .with_columns(
            pl.when(pl.col("z_mediano") > 0)
            .then(pl.lit("medida acima do DMSO"))
            .when(pl.col("z_mediano") < 0)
            .then(pl.lit("medida abaixo do DMSO"))
            .otherwise(pl.lit("mediana sem deslocamento"))
            .alias("direcao")
        )
        .sort("contribuicao_pct", descending=True)
    )

    familias = (
        tabela.group_by("familia")
        .agg(pl.col("contribuicao_pct").sum())
        .sort("contribuicao_pct", descending=True)
    )

    with pl.Config(tbl_rows=top_k, tbl_cols=8, fmt_str_lengths=65):
        display(tabela.head(top_k))
        display(familias)

    return tabela, familias


# Explicar a primeira prioridade, sem atribuir nome farmacológico.
if prioridades.height:
    primeira = prioridades.row(0, named=True)

    print(
        "Condição:",
        primeira[P],
        primeira[EOS],
        primeira[DOSE],
    )

    explicacao_medidas, explicacao_familias = explicar_condicao(
        perfis_dev,
        catalogo_escores,
        placa=primeira[P],
        codigo=primeira[EOS],
        concentracao=primeira[DOSE],
    )

    explicacao_medidas.write_csv(
        PASTA_ESCORES / "explicacao_primeira_prioridade_medidas.csv"
    )

    explicacao_familias.write_csv(
        PASTA_ESCORES / "explicacao_primeira_prioridade_familias.csv"
    )

else:
    print("Não há prioridade exploratória para explicar automaticamente.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import polars as pl

# ============================================================
# 1. Arquivos e configuração
# ============================================================

PASTA_ESCORES = Path(
    "/content/drive/MyDrive/mba_tcc/pipeline_IMTM_HepG2/"
    "20260919T204105_625149Z/"
    "normalizacao_20260919T205516_579360Z/"
    "feature_selection_20260919T211235_656816Z/"
    "escores_exploratorios_20260919T230543_380473Z"
)

arquivos = {
    "resumo": PASTA_ESCORES / "resumo_tratamentos.csv",
    "escores": PASTA_ESCORES / "escores_pocos_R1_R3.parquet",
    "referencias": PASTA_ESCORES / "referencias_dmso.csv",
    "catalogo": PASTA_ESCORES.parent / "catalogo_selecao.csv",
}

for caminho in arquivos.values():
    if not caminho.is_file():
        raise FileNotFoundError(caminho)

P = "Metadata_Plate_ID"
R = "Metadata_Replicate"
EOS = "Metadata_EOS"
DOSE = "Metadata_Concentration"

T = [P, EOS, DOSE]
K = [P, R, "Metadata_Well"]

REPS = ["R1", "R2", "R3"]
N_BOOT = 2000
SEMENTE = 20260919
QUANTIL = 0.95

# Preservar concentração como identificador textual,
# sem arredondar ou misturar condições.
resumo = pl.read_csv(
    arquivos["resumo"],
    schema_overrides={DOSE: pl.String},
)

escores = pl.read_parquet(
    arquivos["escores"]
).with_columns(
    pl.col(DOSE).cast(pl.String)
)

referencias = pl.read_csv(arquivos["referencias"])
catalogo = pl.read_csv(arquivos["catalogo"])

prioridades_fixas = (
    resumo
    .filter(pl.col("prioridade_exploratoria"))
    .with_row_index("ordem_original")
)

if prioridades_fixas.is_empty():
    raise ValueError("A lista de prioridades está vazia.")

if prioridades_fixas.select(T).is_duplicated().any():
    raise ValueError("Há condições repetidas na lista de prioridades.")

if set(escores[R].unique().to_list()) != set(REPS):
    raise ValueError(
        "O arquivo de escores deve conter somente R1-R3."
    )

if escores.select(K).is_duplicated().any():
    raise ValueError("Há poços duplicados nos escores.")

# Reutilizar os blocos principais já definidos.
principais = sorted(
    catalogo.filter(
        (pl.col("status") == "selecionada")
        & (pl.col("uso") == "principal")
    )["bloco"].unique().to_list()
)

if "Mito" not in principais or len(principais) < 2:
    raise ValueError("Faltam blocos principais.")

# Mito fica na posição zero das matrizes.
blocos = ["Mito"] + [b for b in principais if b != "Mito"]
colunas_E = [f"E_{b}" for b in blocos]

if not set(colunas_E).issubset(escores.columns):
    raise ValueError("Faltam escores RMS dos blocos principais.")

# ============================================================
# 2. Função de reamostragem dos controles
# ============================================================

def reamostrar_limites(controles, n_boot, rng):
    """
    Reamostra poços dentro de cada réplica.

    Cada linha contém todos os blocos do mesmo poço.
    Não reamostra cada canal independentemente.
    """
    amostras = [
        x[rng.integers(
            0,
            len(x),
            size=(n_boot, len(x)),
        )]
        for x in controles
    ]

    # Resultado: n_boot × número de blocos.
    return np.quantile(
        np.concatenate(amostras, axis=1),
        QUANTIL,
        axis=1,
        method="linear",
    )


rng = np.random.default_rng(SEMENTE)

limites_boot = {}
limites_originais = {}

registro_limites = []
registro_controles = []

# ============================================================
# 3. Conferir e reamostrar DMSO em cada placa
# ============================================================

for placa in sorted(escores[P].unique().to_list()):
    controles = []

    for replica in REPS:
        tabela = escores.filter(
            (pl.col(P) == placa)
            & (pl.col(R) == replica)
            & (pl.col(EOS) == "DMSO")
        ).sort("Metadata_Well")

        x = tabela.select(colunas_E).to_numpy().astype(float)

        if len(x) < 10 or not np.isfinite(x).all():
            raise ValueError(
                f"DMSO insuficiente ou incompleto: {placa}/{replica}"
            )

        controles.append(x)

    todos = np.concatenate(controles, axis=0)

    limite_original = np.quantile(
        todos,
        QUANTIL,
        axis=0,
        method="linear",
    )

    # Conferir a reprodução do limiar originalmente salvo.
    for j, bloco in enumerate(blocos):
        ref = referencias.filter(
            (pl.col(P) == placa)
            & (pl.col("bloco") == bloco)
        )

        if ref.height != 1:
            raise ValueError(
                f"Referência ausente ou duplicada: {placa}/{bloco}"
            )

        if (
            not np.isclose(ref["quantil"][0], QUANTIL)
            or not np.isclose(
                limite_original[j],
                ref["limiar_RMS"][0],
                rtol=1e-10,
                atol=1e-12,
            )
        ):
            raise ValueError(
                f"O limiar não reproduz o arquivo salvo: "
                f"{placa}/{bloco}"
            )

    qboot = reamostrar_limites(controles, N_BOOT, rng)

    if (
        not np.isfinite(qboot).all()
        or (qboot <= 0).any()
        or (limite_original <= 0).any()
    ):
        raise ValueError(
            f"Referência inválida durante a reamostragem: {placa}"
        )

    limites_boot[placa] = qboot
    limites_originais[placa] = limite_original

    for j, bloco in enumerate(blocos):
        p025, p50, p975 = np.quantile(
            qboot[:, j],
            [0.025, 0.50, 0.975],
            method="linear",
        )

        registro_limites.append({
            P: placa,
            "bloco": bloco,
            "P95_original": float(limite_original[j]),
            "P95_boot_p025": float(p025),
            "P95_boot_mediana": float(p50),
            "P95_boot_p975": float(p975),
        })

        for replica, x in zip(REPS, controles):
            registro_controles.append({
                P: placa,
                R: replica,
                "bloco": bloco,
                "n_dmso": len(x),
                "n_acima_P95_original": int(
                    (x[:, j] > limite_original[j]).sum()
                ),
            })

    print(f"{placa}: controles reamostrados.", flush=True)

# ============================================================
# 4. Verificar somente as prioridades já selecionadas
# ============================================================

resultados = []

for item in prioridades_fixas.iter_rows(named=True):
    placa = item[P]

    condicao = (
        escores.filter(
            (pl.col(P) == placa)
            & (pl.col(EOS) == item[EOS])
            & (pl.col(DOSE) == item[DOSE])
        )
        .group_by(R)
        .agg(*[
            pl.col(c).median().alias(c)
            for c in colunas_E
        ])
        .sort(R)
    )

    if condicao[R].to_list() != REPS:
        raise ValueError(
            f"A condição {item[EOS]} não tem as três réplicas."
        )

    x = condicao.select(colunas_E).to_numpy().astype(float)

    if not np.isfinite(x).all():
        raise ValueError(
            f"Há escore não avaliável em {item[EOS]}."
        )

    cosseno = item["cosseno_mito_min"]

    if cosseno is None or not np.isfinite(cosseno):
        raise ValueError(
            f"Cosseno não avaliável em {item[EOS]}."
        )

    # Conferir que a condição passa pela regra original.
    a_original = x / limites_originais[placa]

    passa_original = (
        ((a_original[:, 0] > 1).sum() >= 2)
        and (a_original[:, 1:] <= 1).all()
        and cosseno > 0
    )

    if not passa_original:
        raise ValueError(
            f"A condição {item[EOS]} não reproduziu a regra original. "
            "Confira se os arquivos pertencem à mesma execução."
        )

    # Dimensões: reamostragem × réplica × bloco.
    a_boot = x[None, :, :] / limites_boot[placa][:, None, :]

    atividade_mito = (
        (a_boot[:, :, 0] > 1).sum(axis=1) >= 2
    )

    # Inclui todos os demais canais e blocos morfológicos.
    demais_abaixo = (
        a_boot[:, :, 1:] <= 1
    ).all(axis=(1, 2))

    permanece = atividade_mito & demais_abaixo & (cosseno > 0)

    resultados.append({
        **{c: item[c] for c in T},
        "permanece_regra_pct": float(100 * permanece.mean()),
        "mantem_atividade_Mito_pct": float(
            100 * atividade_mito.mean()
        ),
        "mantem_demais_abaixo_pct": float(
            100 * demais_abaixo.mean()
        ),
    })

estabilidade = (
    prioridades_fixas
    .join(
        pl.DataFrame(resultados),
        on=T,
        how="left",
        validate="1:1",
    )
    .sort("ordem_original")
)

# ============================================================
# 5. Salvar sem alterar a seleção original
# ============================================================

PASTA_ESTABILIDADE = PASTA_ESCORES / (
    "estabilidade_controles_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
)

PASTA_ESTABILIDADE.mkdir(exist_ok=False)

prioridades_fixas.write_csv(
    PASTA_ESTABILIDADE / "prioridades_originais_congeladas.csv"
)

estabilidade.write_csv(
    PASTA_ESTABILIDADE / "estabilidade_prioridades.csv"
)

pl.DataFrame(registro_limites).write_csv(
    PASTA_ESTABILIDADE / "variacao_limites_DMSO.csv"
)

pl.DataFrame(registro_controles).write_csv(
    PASTA_ESTABILIDADE / "controles_acima_limite_por_replica.csv"
)

manifesto = {
    "n_reamostragens": N_BOOT,
    "semente": SEMENTE,
    "quantil": QUANTIL,
    "replicas": REPS,
    "blocos": blocos,
    "n_prioridades_fixas": prioridades_fixas.height,
    "normalizacao_reajustada": False,
    "selecao_features_reajustada": False,
    "R4_consultada": False,
    "novos_candidatos_pesquisados": False,
    "interpretacao": (
        "Estabilidade condicional aos perfis, normalizacao e features "
        "ja definidos. Nao representa valor-p, FDR ou probabilidade "
        "de seletividade biologica."
    ),
    "sha256_resumo_original": hashlib.sha256(
        arquivos["resumo"].read_bytes()
    ).hexdigest(),
    "numpy": np.__version__,
    "polars": pl.__version__,
}

(PASTA_ESTABILIDADE / "manifesto_estabilidade.json").write_text(
    json.dumps(manifesto, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\nESTABILIDADE DOS CONTROLES CONCLUÍDA")
print("=" * 65)
print(f"Prioridades preservadas: {prioridades_fixas.height}")
print(f"Reamostragens por placa: {N_BOOT}")
print("R4 permanece reservada.")
print(f"Resultados: {PASTA_ESTABILIDADE}")

with pl.Config(tbl_rows=20, tbl_cols=8, fmt_str_lengths=25):
    print(
        estabilidade.select(
            P,
            EOS,
            "A_Mito_mediana",
            "cosseno_mito_min",
            "permanece_regra_pct",
            "mantem_atividade_Mito_pct",
            "mantem_demais_abaixo_pct",
        )
    )

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json
import numpy as np
import polars as pl

from IPython.display import display

# ============================================================
# 1. Arquivos de entrada
# ============================================================

PASTA_EST = Path(
    "/content/drive/MyDrive/mba_tcc/pipeline_IMTM_HepG2/"
    "20260919T204105_625149Z/"
    "normalizacao_20260919T205516_579360Z/"
    "feature_selection_20260919T211235_656816Z/"
    "escores_exploratorios_20260919T230543_380473Z/"
    "estabilidade_controles_20260919T232118_576366Z"
)

PASTA_ESC = PASTA_EST.parent
PASTA_FS = PASTA_ESC.parent

P = "Metadata_Plate_ID"
R = "Metadata_Replicate"
EOS = "Metadata_EOS"
DOSE = "Metadata_Concentration"
COUNT = "Metadata_Object_Count"

T = [P, EOS, DOSE]
K = [P, R, "Metadata_Well"]
REPS = ["R1", "R2", "R3"]

arquivos = {
    "perfis": (
        PASTA_FS / "imtm_hepg2_features_selecionadas.parquet"
    ),
    "catalogo": PASTA_FS / "catalogo_selecao.csv",
    "referencias": PASTA_ESC / "referencias_dmso.csv",
    "prioridades": (
        PASTA_EST / "prioridades_originais_congeladas.csv"
    ),
    "estabilidade": (
        PASTA_EST / "estabilidade_prioridades.csv"
    ),
}

for caminho in arquivos.values():
    if not caminho.is_file():
        raise FileNotFoundError(caminho)

# ============================================================
# 2. Recuperar TODOS os candidatos originais
# ============================================================

prioridades_r4 = pl.read_csv(
    arquivos["prioridades"],
    schema_overrides={DOSE: pl.String},
)

est = pl.read_csv(
    arquivos["estabilidade"],
    schema_overrides={DOSE: pl.String},
)

if (
    prioridades_r4.is_empty()
    or prioridades_r4.select(T).is_duplicated().any()
    or not prioridades_r4["prioridade_exploratoria"].all()
):
    raise ValueError(
        "Lista congelada vazia, repetida ou inconsistente."
    )

prioridades_r4 = (
    prioridades_r4
    .join(
        est.select(T + ["permanece_regra_pct"]),
        on=T,
        how="left",
        validate="1:1",
    )
    .sort("ordem_original")
)

if prioridades_r4["permanece_regra_pct"].null_count():
    raise ValueError(
        "Falta a informação de estabilidade de algum candidato."
    )

# ============================================================
# 3. Recuperar os blocos e os limiares já definidos
# ============================================================

cat = pl.read_csv(arquivos["catalogo"]).filter(
    (pl.col("status") == "selecionada")
    & (pl.col("uso") == "principal")
)

blocos_r4 = {
    bloco: (
        cat.filter(pl.col("bloco") == bloco)
        ["medida"].to_list()
    )
    for bloco in sorted(cat["bloco"].unique().to_list())
}

outros = ["DNA", "ER", "AGP"]

morfologia = [
    bloco
    for bloco in blocos_r4
    if bloco.startswith("Morfologia_")
]

if (
    not {"Mito", *outros}.issubset(blocos_r4)
    or not morfologia
):
    raise ValueError("Faltam blocos principais.")

if set(blocos_r4) != {"Mito", *outros, *morfologia}:
    raise ValueError(
        "Blocos inesperados. Revise antes de abrir R4."
    )

medidas_r4 = [
    coluna
    for colunas in blocos_r4.values()
    for coluna in colunas
]

if len(medidas_r4) != len(set(medidas_r4)):
    raise ValueError("Uma medida aparece em mais de um bloco.")

refs = pl.read_csv(arquivos["referencias"]).filter(
    pl.col("bloco").is_in(list(blocos_r4))
)

if refs.select(P, "bloco").is_duplicated().any():
    raise ValueError("Há referências duplicadas.")

limites = {}

for linha in refs.iter_rows(named=True):
    q = linha["limiar_RMS"]

    if (
        q is None
        or not np.isfinite(q)
        or q <= 0
        or not linha["referencia_utilizavel"]
        or not np.isclose(linha["quantil"], 0.95)
        or linha["n_medidas"] != len(blocos_r4[linha["bloco"]])
    ):
        raise ValueError(
            "Referência original inválida ou incompatível."
        )

    limites[(linha[P], linha["bloco"])] = float(q)

placas_esperadas = (
    set(refs[P].unique().to_list())
    | set(prioridades_r4[P].unique().to_list())
)

for placa in placas_esperadas:
    for bloco in blocos_r4:
        if (placa, bloco) not in limites:
            raise ValueError(
                f"Sem referência para {placa}/{bloco}."
            )

# ============================================================
# 4. Registrar o plano ANTES dos resultados de R4
# ============================================================

def sha256(caminho):
    h = hashlib.sha256()

    with caminho.open("rb") as arquivo:
        for parte in iter(
            lambda: arquivo.read(1024 * 1024),
            b"",
        ):
            h.update(parte)

    return h.hexdigest()


PASTA_R4 = PASTA_ESC / (
    "confirmacao_R4_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
)

PASTA_R4.mkdir(exist_ok=False)

plano_r4 = {
    "registrado_utc": datetime.now(timezone.utc).isoformat(),
    "n_prioridades": prioridades_r4.height,
    "atividade_mito": "A_Mito_R4 > 1",
    "demais_blocos": "todos com A_R4 <= 1",
    "direcao": "cosseno de R4 com CADA replica R1-R3 > 0",
    "limiares": "P95 originais R1-R3, sem reajuste",
    "min_dmso_validos_por_placa": 10,
    "contagem_celular": "apoio; nao e filtro de viabilidade",
    "uso_bootstrap": "descritivo; nenhum candidato retirado",
    "novos_tratamentos_R4": "nao pesquisar",
    "teste_equivalencia": False,
    "correcao_multiplicidade": False,
    "validacao_externa": False,
    "arquivos": {
        nome: {
            "caminho": str(caminho),
            "sha256": sha256(caminho),
        }
        for nome, caminho in arquivos.items()
    },
    "blocos": blocos_r4,
    "numpy": np.__version__,
    "polars": pl.__version__,
}

(PASTA_R4 / "plano_antes_R4.json").write_text(
    json.dumps(
        plano_r4,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    ),
    encoding="utf-8",
)

prioridades_r4.write_csv(
    PASTA_R4 / "candidatos_congelados.csv"
)

print("PLANO DE AVALIAÇÃO REGISTRADO")
print("=" * 65)
print(f"Candidatos mantidos: {prioridades_r4.height}")
print("Nenhum candidato foi retirado pela estabilidade.")
print("Nenhum resultado de R4 foi calculado nesta célula.")
print(f"Pasta: {PASTA_R4}")

In [ ]:
# ============================================================
# 1. Funções numéricas
# ============================================================

def rms_completo(x):
    """RMS sem imputação; uma linha incompleta resulta em NaN."""

    resultado = np.full(len(x), np.nan)
    valido = np.isfinite(x).all(axis=1)

    a = np.abs(x[valido])

    if len(a):
        escala = a.max(axis=1)
        valores = np.zeros(len(a))
        nao_zero = escala > 0

        # Evitar overflow na elevação ao quadrado.
        valores[nao_zero] = escala[nao_zero] * np.sqrt(
            np.mean(
                (a[nao_zero] / escala[nao_zero, None]) ** 2,
                axis=1,
            )
        )

        resultado[valido] = valores

    return resultado


def cosseno_seguro(a, b):
    """Retorna None quando a direção não pode ser estimada."""

    if not np.isfinite(a).all() or not np.isfinite(b).all():
        return None

    escala_a = np.max(np.abs(a))
    escala_b = np.max(np.abs(b))

    if escala_a == 0 or escala_b == 0:
        return None

    a = a / escala_a
    b = b / escala_b

    valor = np.dot(a, b) / (
        np.linalg.norm(a) * np.linalg.norm(b)
    )

    return float(np.clip(valor, -1, 1))


def filtro_condicao(item):
    return pl.all_horizontal([
        pl.col(coluna) == item[coluna]
        for coluna in T
    ])


# ============================================================
# 2. Ler apenas os perfis necessários
# ============================================================

lf = (
    pl.scan_parquet(arquivos["perfis"])
    .select(K + [EOS, DOSE, COUNT] + medidas_r4)
    .with_columns(pl.col(DOSE).cast(pl.String))
)

lista = prioridades_r4.select(T).lazy()

dmso_r4 = (
    lf.filter(
        (pl.col(R) == "R4")
        & (pl.col(EOS) == "DMSO")
    )
    .collect()
)

candidatos_r4 = (
    lf.filter(pl.col(R) == "R4")
    .join(lista, on=T, how="semi")
    .collect()
)

dev_candidatos = (
    lf.filter(pl.col(R).is_in(REPS))
    .join(lista, on=T, how="semi")
    .collect()
)

for tabela in (dmso_r4, candidatos_r4, dev_candidatos):
    if tabela.select(K).is_duplicated().any():
        raise ValueError("Poços duplicados na entrada.")

    if tabela.select(
        pl.any_horizontal([
            pl.col(c).is_null()
            for c in K + [EOS, DOSE]
        ]).any()
    ).item():
        raise ValueError("Há chaves experimentais nulas.")

# ============================================================
# 3. Diagnóstico dos DMSO de R4
# ============================================================

qc_linhas = []
contagem_ref = {}
qc_utilizavel = {}

for placa in sorted(placas_esperadas):
    controles = dmso_r4.filter(pl.col(P) == placa)

    contagens = controles[COUNT].to_numpy().astype(float)
    contagens_validas = contagens[np.isfinite(contagens)]

    contagem_ref[placa] = (
        float(np.median(contagens_validas))
        if len(contagens_validas) else np.nan
    )

    controles_completos = controles.height >= 10

    for bloco, colunas in blocos_r4.items():
        limite = limites[(placa, bloco)]

        e = rms_completo(
            controles.select(colunas).to_numpy().astype(float)
        )

        validos = e[np.isfinite(e)]

        controles_completos = (
            controles_completos
            and len(validos) == controles.height
        )

        qc_linhas.append({
            P: placa,
            "bloco": bloco,
            "n_dmso": controles.height,
            "n_dmso_avaliaveis": len(validos),
            "pct_dmso_acima_limite": (
                float(100 * np.mean(validos > limite))
                if len(validos) else None
            ),

            # Somente diagnóstico.
            # Este P95 NÃO substitui o limiar de R1-R3.
            "P95_R4_sobre_limite_original": (
                float(np.quantile(validos, 0.95) / limite)
                if len(validos) else None
            ),
        })

    qc_utilizavel[placa] = bool(controles_completos)

# ============================================================
# 4. Avaliar cada candidato, preservando a lista original
# ============================================================

resultados_r4 = []

for item in prioridades_r4.iter_rows(named=True):
    placa = item[P]

    d4 = candidatos_r4.filter(filtro_condicao(item))
    d13 = dev_candidatos.filter(filtro_condicao(item))

    linha = {
        **item,
        "n_pocos_R4": d4.height,
    }

    a4 = {}

    # Mesmo escore e mesmos denominadores de R1-R3.
    for bloco, colunas in blocos_r4.items():
        e = rms_completo(
            d4.select(colunas).to_numpy().astype(float)
        )

        a4[bloco] = (
            float(np.median(e) / limites[(placa, bloco)])
            if len(e) and np.isfinite(e).all()
            else None
        )

        linha[f"A_{bloco}_R4"] = a4[bloco]

    completo = (
        d4.height > 0
        and all(
            valor is not None and np.isfinite(valor)
            for valor in a4.values()
        )
    )

    linha["A_outros_max_R4"] = (
        max(a4[b] for b in outros)
        if completo else None
    )

    linha["A_morfologia_max_R4"] = (
        max(a4[b] for b in morfologia)
        if completo else None
    )

    # --------------------------------------------------------
    # Direção: comparar R4 com cada réplica de desenvolvimento
    # --------------------------------------------------------

    mito = blocos_r4["Mito"]

    vetores_dev = (
        d13.group_by(R)
        .agg(*[pl.col(c).median() for c in mito])
        .sort(R)
    )

    if (
        vetores_dev[R].to_list() != REPS
        or not np.isfinite(d13.select(mito).to_numpy()).all()
    ):
        raise ValueError(
            f"Referência R1-R3 inconsistente para {item[EOS]}."
        )

    cossenos = [None] * 3

    if d4.height:
        x4 = d4.select(mito).to_numpy().astype(float)

        if np.isfinite(x4).all():
            vetor_r4 = np.median(x4, axis=0)

            cossenos = [
                cosseno_seguro(vetor_r4, vetor)
                for vetor in vetores_dev.select(mito).to_numpy()
            ]

    for replica, valor in zip(REPS, cossenos):
        linha[f"cosseno_R4_{replica}"] = valor

    cosseno_min = (
        min(cossenos)
        if all(valor is not None for valor in cossenos)
        else None
    )

    linha["cosseno_min_R4_dev"] = cosseno_min

    # --------------------------------------------------------
    # Contagem celular relativa: apoio, não filtro
    # --------------------------------------------------------

    contagens_r4 = d4[COUNT].to_numpy().astype(float)
    referencia_contagem = contagem_ref.get(placa, np.nan)

    linha["contagem_relativa_R4"] = (
        float(np.median(contagens_r4) / referencia_contagem)
        if (
            len(contagens_r4)
            and np.isfinite(contagens_r4).all()
            and np.isfinite(referencia_contagem)
            and referencia_contagem > 0
        )
        else None
    )

    # --------------------------------------------------------
    # Classificação conforme a regra registrada
    # --------------------------------------------------------

    linha["repete_regra_R4"] = None
    linha["motivos"] = ""

    if not d4.height:
        linha["status_R4"] = "inconclusivo_sem_perfil"

    elif not completo or not qc_utilizavel.get(placa, False):
        linha["status_R4"] = "inconclusivo_dados_ou_controles"

    elif cosseno_min is None:
        linha["status_R4"] = "inconclusivo_direcao"

    else:
        motivos = []

        if a4["Mito"] <= 1:
            motivos.append("Mito_nao_ultrapassou_referencia")

        if linha["A_outros_max_R4"] > 1:
            motivos.append("outro_canal_acima_da_referencia")

        if linha["A_morfologia_max_R4"] > 1:
            motivos.append("morfologia_acima_da_referencia")

        if cosseno_min <= 0:
            motivos.append(
                "direcao_nao_concordante_com_todas_replicas"
            )

        linha["repete_regra_R4"] = not motivos

        linha["status_R4"] = (
            "padrao_repetido"
            if not motivos
            else "padrao_nao_repetido"
        )

        linha["motivos"] = "; ".join(motivos)

    resultados_r4.append(linha)

    print(
        f"{placa} / {item[EOS]}: {linha['status_R4']}",
        flush=True,
    )

confirmacao_r4 = (
    pl.DataFrame(
        resultados_r4,
        infer_schema_length=None,
        schema_overrides={"repete_regra_R4": pl.Boolean},
    )
    .sort("ordem_original")
)

qc_dmso_r4 = pl.DataFrame(
    qc_linhas,
    infer_schema_length=None,
)

if confirmacao_r4.height != prioridades_r4.height:
    raise RuntimeError(
        "O número de candidatos foi alterado durante a avaliação."
    )

In [ ]:
confirmacao_r4.write_csv(
    PASTA_R4 / "confirmacao_12_candidatos_R4.csv"
)

confirmacao_r4.write_parquet(
    PASTA_R4 / "confirmacao_R4.parquet"
)

qc_dmso_r4.write_csv(
    PASTA_R4 / "diagnostico_DMSO_R4.csv"
)

resumo_r4 = (
    confirmacao_r4
    .group_by("status_R4")
    .len()
    .sort("status_R4")
)

(PASTA_R4 / "resumo_execucao.json").write_text(
    json.dumps(
        {
            "status": resumo_r4.to_dicts(),
            "n_candidatos": confirmacao_r4.height,
            "nenhuma_recalibracao": True,
            "R4_agora_consultada": True,
        },
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    ),
    encoding="utf-8",
)

print("\nAVALIAÇÃO DA RÉPLICA R4 CONCLUÍDA")
print("=" * 65)
print(f"Candidatos acompanhados: {confirmacao_r4.height}")
print(f"Resultados: {PASTA_R4}")

display(resumo_r4)

with pl.Config(tbl_rows=20, tbl_cols=9, fmt_str_lengths=32):
    display(
        confirmacao_r4.select(
            EOS,
            "permanece_regra_pct",
            "A_Mito_R4",
            "A_outros_max_R4",
            "A_morfologia_max_R4",
            "cosseno_min_R4_dev",
            "status_R4",
        )
    )

print("\nDIAGNÓSTICO DOS DMSO DE R4")

with pl.Config(tbl_rows=55, tbl_cols=6, fmt_str_lengths=26):
    display(qc_dmso_r4)

print("\nOs arquivos anteriores permanecem intactos.")
print("Não foram pesquisados novos candidatos em R4.")

In [ ]:
%pip install -q "polars>=1.0,<2" "scikit-learn>=1.3,<2" threadpoolctl

from pathlib import Path
from datetime import datetime, timezone
from itertools import permutations

import hashlib
import json
import numpy as np
import polars as pl
import sklearn

from sklearn.covariance import LedoitWolf
from threadpoolctl import threadpool_limits
from IPython.display import display

# ============================================================
# 1. Arquivos e configuração
# ============================================================

PASTA_FS = Path(
    "/content/drive/MyDrive/mba_tcc/pipeline_IMTM_HepG2/"
    "20260919T204105_625149Z/"
    "normalizacao_20260919T205516_579360Z/"
    "feature_selection_20260919T211235_656816Z"
)

ARQ_PERFIS = (
    PASTA_FS / "imtm_hepg2_features_selecionadas.parquet"
)
ARQ_CATALOGO = PASTA_FS / "catalogo_selecao.csv"

P = "Metadata_Plate_ID"
R = "Metadata_Replicate"
EOS = "Metadata_EOS"

K = [P, R, "Metadata_Well"]

REPS = ["R1", "R2", "R3"]
QUANTIL = 0.95

# Proteção numérica adicional aplicada aos autovalores.
# Não é um limiar de atividade biológica.
PISO_RELATIVO = 1e-8

REPS_AJUSTE_FINAL = ["R1", "R2"]
REP_CAL_FINAL = "R3"

for arquivo in [ARQ_PERFIS, ARQ_CATALOGO]:
    if not arquivo.is_file():
        raise FileNotFoundError(arquivo)

# ============================================================
# 2. Reutilizar os blocos já classificados
# ============================================================

catalogo = pl.read_csv(ARQ_CATALOGO).filter(
    (pl.col("status") == "selecionada")
    & (pl.col("uso") == "principal")
)

blocos = {
    b: catalogo.filter(
        pl.col("bloco") == b
    )["medida"].to_list()
    for b in sorted(catalogo["bloco"].unique().to_list())
}

principais = [
    c for cols in blocos.values() for c in cols
]

esperados = {
    "Mito", "DNA", "ER", "AGP",
    "Morfologia_Cells",
    "Morfologia_Cyto",
    "Morfologia_Nuc",
}

if (
    set(blocos) != esperados
    or len(principais) != len(set(principais))
):
    raise ValueError(
        "Blocos diferentes dos esperados ou medidas repetidas."
    )

# ============================================================
# 3. Ler apenas R1-R3
# ============================================================

lf = pl.scan_parquet(ARQ_PERFIS)
schema = lf.collect_schema()

metas = [
    c for c in schema.names()
    if c.startswith("Metadata_")
]

if not set(K + [EOS] + principais).issubset(schema.names()):
    raise ValueError("Colunas necessárias ausentes no Parquet.")

if any(not schema[c].is_numeric() for c in principais):
    raise TypeError("Todas as medidas precisam ser numéricas.")

base_sph = (
    lf
    .filter(pl.col(R).is_in(REPS))
    .select(metas + principais)
    .collect()
    .sort(K)
)

if (
    base_sph.is_empty()
    or base_sph.select(K).is_duplicated().any()
):
    raise ValueError("Base vazia ou poços repetidos.")

if base_sph.select(
    pl.any_horizontal([
        pl.col(c).is_null()
        for c in K + [EOS]
    ]).any()
).item():
    raise ValueError(
        "Chaves ou identificação de controle ausentes."
    )

# ============================================================
# 4. Conferir o equilíbrio dos controles
# ============================================================

controles_sph = base_sph.filter(
    pl.col(EOS) == "DMSO"
)

contagens = (
    controles_sph
    .group_by(P, R)
    .len()
    .sort([P, R])
)

placas = sorted(base_sph[P].unique().to_list())

if set(contagens.select(P, R).rows()) != {
    (p, r) for p in placas for r in REPS
}:
    raise ValueError(
        "Existe grupo placa/réplica sem DMSO."
    )

if (
    contagens["len"].min() < 10
    or contagens["len"].n_unique() != 1
):
    raise ValueError(
        "Controles insuficientes ou desbalanceados entre grupos."
    )

rep_ctrl = controles_sph[R].to_numpy()
placa_ctrl = controles_sph[P].to_numpy()

print("ENTRADA DO TESTE")
print("=" * 65)
print(f"Poços R1-R3: {base_sph.height:,}")
print(f"DMSO R1-R3: {controles_sph.height:,}")
print(
    f"Medidas principais: {len(principais):,}; "
    f"blocos: {len(blocos)}"
)
print("R4 não entra nesta execução.")

In [ ]:
def ajustar_sphering(x: np.ndarray) -> dict:
    """ZCA regularizado; x deve conter somente DMSO de ajuste."""

    x = np.asarray(x, dtype=np.float64)

    if (
        x.ndim != 2
        or x.shape[0] < 10
        or x.shape[1] < 1
    ):
        raise ValueError(
            "Matriz de ajuste vazia ou com poucos controles."
        )

    if not np.isfinite(x).all():
        raise ValueError(
            "DMSO de ajuste com ausências ou infinitos."
        )

    estimador = LedoitWolf(
        store_precision=False
    ).fit(x)

    cov = estimador.covariance_
    cov = (cov + cov.T) / 2.0

    autovalores, vetores = np.linalg.eigh(cov)
    maior = float(autovalores[-1])

    if not np.isfinite(maior) or maior <= 0:
        raise ValueError(
            "Covariância sem variação utilizável."
        )

    # Proteção contra inversão de direções quase singulares.
    piso = maior * PISO_RELATIVO
    usados = np.maximum(autovalores, piso)

    w = (
        vetores * (1.0 / np.sqrt(usados))
    ) @ vetores.T

    cov_usada = (vetores * usados) @ vetores.T

    erro = float(np.max(np.abs(
        w @ cov_usada @ w.T - np.eye(x.shape[1])
    )))

    if not np.isfinite(w).all() or erro > 1e-6:
        raise RuntimeError(
            "Falha na verificação numérica do sphering."
        )

    return {
        "centro": estimador.location_,
        "W": w,
        "cov_lw": cov,
        "cov_usada": cov_usada,
        "autovalores": autovalores,
        "autovalores_usados": usados,
        "shrinkage": float(estimador.shrinkage_),
        "n_piso": int(np.sum(autovalores < piso)),
        "condicao": float(usados[-1] / usados[0]),
        "erro_identidade": erro,
    }


def transformar_sphering(
    x: np.ndarray,
    modelo: dict,
) -> np.ndarray:
    """
    Uma linha incompleta fica inteiramente não avaliável
    naquele bloco. Não há imputação.
    """

    x = np.asarray(x, dtype=np.float64)

    if (
        x.ndim != 2
        or x.shape[1] != len(modelo["centro"])
    ):
        raise ValueError(
            "Número de medidas incompatível com o modelo."
        )

    y = np.full(x.shape, np.nan)
    ok = np.isfinite(x).all(axis=1)

    y[ok] = (
        x[ok] - modelo["centro"]
    ) @ modelo["W"]

    if not np.isfinite(y[ok]).all():
        raise FloatingPointError(
            "Overflow ao transformar valores finitos."
        )

    return y


def rms_seguro(x: np.ndarray) -> np.ndarray:
    """RMS sem tratar ausências como zero."""

    x = np.asarray(x, dtype=np.float64)
    e = np.full(len(x), np.nan)

    ok = np.isfinite(x).all(axis=1)
    a = np.abs(x[ok])

    if len(a):
        escala = a.max(axis=1)
        valores = np.zeros(len(a))
        nz = escala > 0

        valores[nz] = escala[nz] * np.sqrt(
            np.mean(
                (a[nz] / escala[nz, None]) ** 2,
                axis=1,
            )
        )

        e[ok] = valores

    return e


def correlacao_absoluta_media(x: np.ndarray):
    """Correlação de Pearson absoluta, fora da diagonal."""

    validas = np.std(x, axis=0) > 0

    if validas.sum() < 2:
        return None

    r = np.corrcoef(
        x[:, validas],
        rowvar=False,
    )

    valores = np.abs(
        r[np.triu_indices(len(r), 1)]
    )
    valores = valores[np.isfinite(valores)]

    return (
        float(valores.mean())
        if len(valores) else None
    )

In [ ]:
avaliacoes = []
por_placa = []

# Limitar as threads evita excesso de paralelismo
# em muitas operações matriciais pequenas.
with threadpool_limits(limits=1):

    for bloco, cols in blocos.items():
        c = (
            controles_sph.select(cols)
            .to_numpy()
            .astype(np.float64)
        )

        if not np.isfinite(c).all():
            raise ValueError(
                f"Controles incompletos em {bloco}."
            )

        # Cada réplica fornece um modelo de ajuste.
        modelos_rotacao = {
            r: ajustar_sphering(c[rep_ctrl == r])
            for r in REPS
        }

        for ajuste, calibracao, teste in permutations(REPS):
            modelo = modelos_rotacao[ajuste]

            mcal = rep_ctrl == calibracao
            mtest = rep_ctrl == teste

            for metodo in ["MAD", "MAD_ZCA_LW"]:
                y = (
                    c
                    if metodo == "MAD"
                    else transformar_sphering(c, modelo)
                )

                ecal = rms_seguro(y[mcal])
                etest = rms_seguro(y[mtest])

                # Um P95 por bloco, comum às placas,
                # estimado somente na réplica de calibração.
                limite = float(np.quantile(
                    ecal,
                    QUANTIL,
                    method="linear",
                ))

                if not np.isfinite(limite) or limite <= 0:
                    raise ValueError(
                        "Limiar de calibração não utilizável."
                    )

                info = {
                    "bloco": bloco,
                    "metodo": metodo,
                    "rep_ajuste": ajuste,
                    "rep_calibracao": calibracao,
                    "rep_teste": teste,
                }

                taxas = []
                medianas = []

                for placa in placas:
                    valores = etest[
                        placa_ctrl[mtest] == placa
                    ]

                    taxa = float(
                        100 * np.mean(valores > limite)
                    )
                    mediana = float(
                        np.median(valores) / limite
                    )

                    taxas.append(taxa)
                    medianas.append(mediana)

                    por_placa.append({
                        **info,
                        P: placa,
                        "n_dmso_teste": len(valores),
                        "P95_calibracao": limite,
                        "pct_acima_P95": taxa,
                        "mediana_sobre_P95": mediana,
                    })

                media_med = float(np.mean(medianas))

                avaliacoes.append({
                    **info,
                    "n_dmso_teste": int(mtest.sum()),

                    "taxa_acima_P95_pct": float(
                        100 * np.mean(etest > limite)
                    ),

                    "erro_placas_pp": float(
                        np.mean(np.abs(
                            np.array(taxas) - 5.0
                        ))
                    ),

                    "cv_medianas_placas": (
                        float(np.std(medianas) / media_med)
                        if media_med > 0 else None
                    ),

                    "corr_abs_teste": (
                        correlacao_absoluta_media(y[mtest])
                    ),
                })

        print(
            f"{bloco}: seis combinações concluídas.",
            flush=True,
        )

avaliacao_sph = pl.DataFrame(avaliacoes)
controle_placas_sph = pl.DataFrame(por_placa)

comparacao_sph = (
    avaliacao_sph
    .group_by("bloco", "metodo")
    .agg(
        pl.col("taxa_acima_P95_pct")
        .mean()
        .alias("taxa_media_pct"),

        pl.col("erro_placas_pp")
        .mean()
        .alias("erro_placas_pp"),

        pl.col("cv_medianas_placas")
        .mean()
        .alias("cv_placas"),

        pl.col("corr_abs_teste")
        .mean()
        .alias("corr_abs"),
    )
    .sort(["bloco", "metodo"])
)

print("\nCOMPARAÇÃO NOS CONTROLES")
with pl.Config(tbl_rows=20, tbl_cols=8):
    display(comparacao_sph)

In [ ]:
# ============================================================
# 1. Nova pasta, sem sobrescrever análises anteriores
# ============================================================

PASTA_SPH = PASTA_FS / (
    "teste_sphering_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
)

PASTA_SPH.mkdir(exist_ok=False)
(PASTA_SPH / "modelos").mkdir()

df_imtm_sph = base_sph.select(metas)
escores_sph = base_sph.select(metas)

modelos_sph = {}
mapa_coordenadas = {}
parametros = []

# ============================================================
# 2. Ajustar em R1-R2 e calibrar em R3
# ============================================================

with threadpool_limits(limits=1):

    for bloco, cols in blocos.items():
        c = (
            controles_sph.select(cols)
            .to_numpy()
            .astype(np.float64)
        )
        x = (
            base_sph.select(cols)
            .to_numpy()
            .astype(np.float64)
        )

        mfit = np.isin(rep_ctrl, REPS_AJUSTE_FINAL)
        mcal = rep_ctrl == REP_CAL_FINAL

        modelo = ajustar_sphering(c[mfit])
        y = transformar_sphering(x, modelo)

        q_mad = float(np.quantile(
            rms_seguro(c[mcal]),
            QUANTIL,
            method="linear",
        ))

        q_sph = float(np.quantile(
            rms_seguro(
                transformar_sphering(c[mcal], modelo)
            ),
            QUANTIL,
            method="linear",
        ))

        if (
            not np.isfinite([q_mad, q_sph]).all()
            or min(q_mad, q_sph) <= 0
        ):
            raise ValueError(
                "Limiar final de calibração não utilizável."
            )

        coordenadas = [
            f"SPH_{bloco}_{i:04d}"
            for i in range(1, len(cols) + 1)
        ]

        df_imtm_sph = df_imtm_sph.hstack(
            pl.from_numpy(
                y,
                schema=coordenadas,
            ).fill_nan(None)
        )

        # Escores experimentais com limiares globais por bloco.
        # Não são os mesmos A_k da análise anterior.
        escores_sph = escores_sph.with_columns(
            pl.Series(
                f"E_MAD_{bloco}", rms_seguro(x)
            ).fill_nan(None),

            pl.Series(
                f"E_SPH_{bloco}", rms_seguro(y)
            ).fill_nan(None),

            pl.Series(
                f"A_MAD_global_{bloco}",
                rms_seguro(x) / q_mad,
            ).fill_nan(None),

            pl.Series(
                f"A_SPH_global_{bloco}",
                rms_seguro(y) / q_sph,
            ).fill_nan(None),
        )

        modelos_sph[bloco] = modelo

        mapa_coordenadas[bloco] = {
            "medidas_entrada": cols,
            "coordenadas_saida": coordenadas,
        }

        # Salvar matrizes e ordem das medidas, sem pickle.
        destino_modelo = (
            PASTA_SPH / "modelos" / f"{bloco}.npz"
        )

        np.savez_compressed(
            destino_modelo,
            **modelo,
            P95_cal_MAD=q_mad,
            P95_cal_SPH=q_sph,
            medidas=np.asarray(cols, dtype=str),
            coordenadas=np.asarray(coordenadas, dtype=str),
        )

        with np.load(
            destino_modelo,
            allow_pickle=False,
        ) as salvo:
            np.testing.assert_array_equal(
                salvo["W"], modelo["W"]
            )
            np.testing.assert_array_equal(
                salvo["centro"], modelo["centro"]
            )

        parametros.append({
            "bloco": bloco,
            "n_medidas": len(cols),
            "n_dmso_ajuste": int(mfit.sum()),
            "n_dmso_cal": int(mcal.sum()),
            "P95_cal_MAD": q_mad,
            "P95_cal_SPH": q_sph,
            "shrinkage": modelo["shrinkage"],
            "condicao_cov": modelo["condicao"],
            "n_autovalores_no_piso": modelo["n_piso"],
            "erro_identidade": modelo["erro_identidade"],
            "pocos_nao_avaliaveis": int(
                (~np.isfinite(y).all(axis=1)).sum()
            ),
        })

parametros_sph = pl.DataFrame(parametros)

if not df_imtm_sph.select(metas).equals(
    base_sph.select(metas)
):
    raise RuntimeError(
        "Metadados ou ordem dos poços foram alterados."
    )

# ============================================================
# 3. Salvar as tabelas e conferir a gravação
# ============================================================

for nome, tabela in {
    "perfis_ZCA_LW_R1_R3": df_imtm_sph,
    "escores_calibrados_R1_R3": escores_sph,
}.items():

    destino = PASTA_SPH / f"{nome}.parquet"

    tabela.write_parquet(
        destino,
        compression="zstd",
    )

    if not pl.read_parquet(destino).equals(tabela):
        raise RuntimeError(
            f"Falha no salvamento de {nome}."
        )

for nome, tabela in {
    "comparacao_controles": comparacao_sph,
    "avaliacao_por_rotacao": avaliacao_sph,
    "controles_por_placa_rotacao": controle_placas_sph,
    "parametros_modelos": parametros_sph,
}.items():
    tabela.write_csv(PASTA_SPH / f"{nome}.csv")

controles_sph.select(K).with_columns(
    pl.when(pl.col(R).is_in(REPS_AJUSTE_FINAL))
    .then(pl.lit("ajuste"))
    .otherwise(pl.lit("calibracao"))
    .alias("papel_modelo_final")
).write_csv(
    PASTA_SPH / "pocos_DMSO_papeis.csv"
)

(PASTA_SPH / "mapa_coordenadas.json").write_text(
    json.dumps(
        mapa_coordenadas,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

# ============================================================
# 4. Registrar a procedência e as decisões
# ============================================================

def hash_arquivo(caminho):
    h = hashlib.sha256()

    with caminho.open("rb") as f:
        for pedaco in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(pedaco)

    return h.hexdigest()


manifesto_sph = {
    "tipo": (
        "comparacao_pos_hoc_condicional_a_MAD_e_features_fixos"
    ),
    "metodo": (
        "ZCA por bloco + LedoitWolf + piso relativo de autovalores"
    ),
    "rotacoes": list(permutations(REPS)),
    "ordem_rotacoes": [
        "ajuste", "calibracao", "avaliacao"
    ],
    "controle": "Metadata_EOS == 'DMSO'",
    "quantil_teste": QUANTIL,
    "P95_teste": (
        "comum as sete placas; somente replica de calibracao"
    ),
    "piso_relativo": PISO_RELATIVO,
    "ajuste_final": REPS_AJUSTE_FINAL,
    "calibracao_final": REP_CAL_FINAL,
    "R4_utilizada_nesta_execucao": False,
    "R4_ja_consultada_em_analise_anterior": True,
    "limiares_do_modelo_final": (
        "P95 dos DMSO de R3; comuns as placas de cada bloco"
    ),
    "ranqueamento_realizado": False,
    "farmacologia_utilizada": False,
    "n_pocos": base_sph.height,
    "n_dmso": controles_sph.height,
    "n_medidas": len(principais),
    "sha256_perfis": hash_arquivo(ARQ_PERFIS),
    "sha256_catalogo": hash_arquivo(ARQ_CATALOGO),
    "versoes": {
        "numpy": np.__version__,
        "polars": pl.__version__,
        "scikit_learn": sklearn.__version__,
    },
}

(PASTA_SPH / "manifesto_sphering.json").write_text(
    json.dumps(
        manifesto_sph,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    ),
    encoding="utf-8",
)

# ============================================================
# 5. Mostrar os resultados
# ============================================================

print("\nTESTE DE SPHERING CONCLUÍDO")
print("=" * 65)
print(
    f"Poços preservados em R1-R3: {df_imtm_sph.height:,}"
)
print(
    "DMSO no ajuste final: "
    f"{int(np.isin(rep_ctrl, REPS_AJUSTE_FINAL).sum()):,}"
)
print(
    "DMSO na calibração final: "
    f"{int((rep_ctrl == REP_CAL_FINAL).sum()):,}"
)
print(
    f"Blocos: {len(blocos)}; "
    f"medidas principais: {len(principais)}"
)
print(f"Resultados: {PASTA_SPH}")

with pl.Config(
    tbl_rows=20,
    tbl_cols=8,
    fmt_str_lengths=30,
):
    display(comparacao_sph)

    display(
        parametros_sph.select(
            "bloco",
            "n_medidas",
            "shrinkage",
            "condicao_cov",
            "n_autovalores_no_piso",
            "P95_cal_SPH",
            "pocos_nao_avaliaveis",
        )
    )

print("\nR4 não foi usada nesta execução.")
print("Os resultados anteriores permanecem intactos.")
print(
    "Os escores globais são experimentais; "
    "nenhum candidato foi reclassificado."
)